# AI-Assisted Diagnosis of Cutaneous Melanoma

## Complete dermoscopic segmentation, classification, explainability, and demonstration pipeline

This Google Colab notebook implements the proposed biomedical-engineering workflow:

**Dermoscopic image → preprocessing → lesion segmentation → padded lesion ROI → EfficientNetB0 classification → model probability of malignancy → Grad-CAM → Gradio application**

Two segmentation models are trained under equivalent conditions:

1. Standard U-Net
2. Attention U-Net

The final segmenter is selected using **validation Dice and IoU**, not test performance. The classifier uses a patient-level split of the labelled SIIM-ISIC 2020 training set, removes the official duplicate-image entries, handles class imbalance, selects its decision threshold only from validation data, and is evaluated once on the held-out test set.

> **Research/educational prototype only. This notebook and its web interface are not medical diagnoses and must not replace assessment by a qualified healthcare professional.**


## How to run this notebook

1. In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another GPU).
2. Run the setup and configuration cells.
3. Leave `RUN_MODE = "quick"` while checking that the complete workflow runs.
4. For the final university experiment, change `RUN_MODE` to `"full"`, restart the runtime, and run all cells.
5. The large official archives are kept in Google Drive. Working data and ROI crops can be placed in `/content` for faster training.
6. Trained models, manifests, histories, thresholds, tables, plots, and deployment files are saved under `MyDrive/Melanoma_AI_Project`.

### Expected long-running stages

- Download/extraction: ISIC 2018 Task 1 and SIIM-ISIC 2020 JPEG training archives.
- Segmentation: Standard U-Net and Attention U-Net training.
- ROI generation: one batched pass of the selected segmenter over the 2020 images.
- Classification: frozen-head training followed by partial EfficientNetB0 fine-tuning.
- Optional ablation: a second EfficientNetB0 trained on full images rather than lesion ROIs.

The notebook does not load either complete dataset into a single NumPy array. It uses file manifests, `tf.data`, batches, and prefetching.


## 1. Environment setup


In [ ]:
# Colab already includes TensorFlow. Install only the lightweight extras used here.
import importlib.util
import subprocess
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "gradio>=5.0,<7", "scipy>=1.10", "tqdm>=4.66", "requests>=2.31", "socksio>=1.0"
    ])
else:
    print("Local environment detected. Install TensorFlow and the listed packages if needed.")


## 2. Global configuration


In [ ]:
# Change only RUN_MODE to "full" for the final experiment.
RUN_MODE = "quick"                     # "quick" or "full"
SEED = 42

SEG_IMAGE_SIZE = (256, 256)
CLS_IMAGE_SIZE = (224, 224)
SEG_BATCH_SIZE = 8
CLS_BATCH_SIZE = 32

SEG_EPOCHS = 50
CLS_HEAD_EPOCHS = 10
CLS_FINE_TUNE_EPOCHS = 20

QUICK_SEG_CASES = 128
QUICK_CLS_MAX_IMAGES = 1200

SEG_THRESHOLD = 0.50
ROI_PADDING_FRACTION = 0.10
MIN_VALID_MASK_FRACTION = 0.001
MAX_VALID_MASK_FRACTION = 0.95

SEG_LEARNING_RATE = 1e-3
CLS_HEAD_LEARNING_RATE = 1e-3
CLS_FINE_TUNE_LEARNING_RATE = 1e-5
CLS_FINE_TUNE_LAST_N = 40

USE_MIXED_PRECISION = True
VERIFY_IMAGE_CONTENT = True
RUN_ABLATION = False                  # True compares full-image vs ROI classifiers
ROI_STORAGE = "runtime"              # "runtime" (faster) or "drive" (persistent)
LAUNCH_GRADIO = True

EXPECTED_SEGMENTATION_CASES = 2594
EXPECTED_CLASSIFICATION_IMAGES = 33126
EXPECTED_DUPLICATE_PAIRS = 425

if RUN_MODE not in {"quick", "full"}:
    raise ValueError('RUN_MODE must be "quick" or "full".')
if ROI_STORAGE not in {"runtime", "drive"}:
    raise ValueError('ROI_STORAGE must be "runtime" or "drive".')

print("RUN_MODE:", RUN_MODE)
if RUN_MODE != "full":
    print("QUICK MODE is for debugging only. Use FULL mode for final reported results.")


## 3. Imports, reproducibility, GPU, and mixed precision


In [ ]:
import ast
import gc
import json
import math
import os
import random
import re
import shutil
import time
import warnings
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import tensorflow as tf
from PIL import Image, UnidentifiedImageError
from scipy import ndimage
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow import keras
from tensorflow.keras import layers
from tqdm.auto import tqdm

try:
    from IPython.display import display
except ImportError:
    display = print

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    print("TensorFlow deterministic operations enabled.")
except Exception as exc:
    print("Deterministic operations could not be fully enabled:", exc)

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPU devices:", gpus)

MIXED_PRECISION_ENABLED = False
if gpus and USE_MIXED_PRECISION:
    try:
        details = tf.config.experimental.get_device_details(gpus[0])
        capability = details.get("compute_capability", (0, 0))
        if tuple(capability) >= (7, 0):
            keras.mixed_precision.set_global_policy("mixed_float16")
            MIXED_PRECISION_ENABLED = True
    except Exception as exc:
        print("Mixed precision was not enabled:", exc)

print("Keras numeric policy:", keras.mixed_precision.global_policy())
print("Mixed precision enabled:", MIXED_PRECISION_ENABLED)


## 4. Google Drive and project folders


In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/Melanoma_AI_Project")
    RUNTIME_ROOT = Path("/content/melanoma_ai_work")
else:
    PROJECT_DIR = Path(os.environ.get("MELANOMA_PROJECT_DIR", "./Melanoma_AI_Project")).resolve()
    RUNTIME_ROOT = PROJECT_DIR / "local_work"

ARCHIVE_DIR = PROJECT_DIR / "archives"
RESULTS_DIR = PROJECT_DIR / "results"
MODELS_DIR = PROJECT_DIR / "models"
MANIFEST_DIR = PROJECT_DIR / "manifests"
DEPLOYMENT_DIR = PROJECT_DIR / "huggingface_space"
SEG_WORK_DIR = RUNTIME_ROOT / "isic2018_segmentation"
CLS_WORK_DIR = RUNTIME_ROOT / "siim_isic2020_classification"
ROI_DIR = (RUNTIME_ROOT / "classification_rois") if ROI_STORAGE == "runtime" else (PROJECT_DIR / "classification_rois")

for folder in [PROJECT_DIR, ARCHIVE_DIR, RESULTS_DIR, MODELS_DIR, MANIFEST_DIR,
               DEPLOYMENT_DIR, RUNTIME_ROOT, SEG_WORK_DIR, CLS_WORK_DIR, ROI_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "run_mode": RUN_MODE,
    "seed": SEED,
    "seg_image_size": list(SEG_IMAGE_SIZE),
    "cls_image_size": list(CLS_IMAGE_SIZE),
    "seg_batch_size": SEG_BATCH_SIZE,
    "cls_batch_size": CLS_BATCH_SIZE,
    "seg_epochs": SEG_EPOCHS,
    "cls_head_epochs": CLS_HEAD_EPOCHS,
    "cls_fine_tune_epochs": CLS_FINE_TUNE_EPOCHS,
    "seg_threshold": SEG_THRESHOLD,
    "roi_padding_fraction": ROI_PADDING_FRACTION,
    "mixed_precision": MIXED_PRECISION_ENABLED,
    "run_ablation": RUN_ABLATION,
    "roi_storage": ROI_STORAGE,
}

(RESULTS_DIR / "pipeline_config.json").write_text(
    json.dumps(CONFIG, indent=2), encoding="utf-8"
)
print("Project directory:", PROJECT_DIR)
print("Working directory:", RUNTIME_ROOT)


## 5. Official dataset sources and restart-safe downloads

The URLs below are the direct files linked from the official [ISIC Challenge datasets page](https://challenge.isic-archive.com/data/). The 2018 Task 1 inputs contain 2,594 training images; the corresponding ground-truth archive contains their binary masks. The SIIM-ISIC 2020 labelled training data contain 33,126 JPEG images. The v2 metadata add lesion identifiers, and the official duplicates CSV identifies 425 duplicate pairs.

If an automatic transfer fails, the cell stops and prints the exact filename to place in the project's `archives` folder. A valid existing file is never downloaded again. Partial downloads use a `.part` file and an HTTP Range request when the server supports it.


In [ ]:
DATASET_SPECS = {
    "seg_images": {
        "filename": "ISIC2018_Task1-2_Training_Input.zip",
        "url": "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task1-2_Training_Input.zip",
        "kind": "zip",
    },
    "seg_masks": {
        "filename": "ISIC2018_Task1_Training_GroundTruth.zip",
        "url": "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task1_Training_GroundTruth.zip",
        "kind": "zip",
    },
    "cls_images": {
        "filename": "ISIC_2020_Training_JPEG.zip",
        "url": "https://isic-archive.s3.amazonaws.com/challenges/2020/ISIC_2020_Training_JPEG.zip",
        "kind": "zip",
    },
    "cls_ground_truth": {
        "filename": "ISIC_2020_Training_GroundTruth.csv",
        "url": "https://isic-archive.s3.amazonaws.com/challenges/2020/ISIC_2020_Training_GroundTruth.csv",
        "kind": "csv",
    },
    "cls_metadata_v2": {
        "filename": "ISIC_2020_Training_GroundTruth_v2.csv",
        "url": "https://isic-archive.s3.amazonaws.com/challenges/2020/ISIC_2020_Training_GroundTruth_v2.csv",
        "kind": "csv",
    },
    "cls_duplicates": {
        "filename": "ISIC_2020_Training_Duplicates.csv",
        "url": "https://isic-archive.s3.amazonaws.com/challenges/2020/ISIC_2020_Training_Duplicates.csv",
        "kind": "csv",
    },
}


def file_is_valid(path, kind):
    path = Path(path)
    if not path.exists() or path.stat().st_size < 100:
        return False
    if kind == "zip":
        return zipfile.is_zipfile(path)
    if kind == "csv":
        try:
            return len(pd.read_csv(path, nrows=3).columns) >= 1
        except Exception:
            return False
    return True


def remote_file_size(url, timeout=30):
    try:
        response = requests.head(url, allow_redirects=True, timeout=timeout)
        response.raise_for_status()
        value = response.headers.get("Content-Length")
        return int(value) if value else None
    except Exception:
        return None


def download_with_resume(spec, destination_dir=ARCHIVE_DIR, chunk_mb=8):
    destination_dir = Path(destination_dir)
    destination_dir.mkdir(parents=True, exist_ok=True)
    final_path = destination_dir / spec["filename"]
    part_path = final_path.with_suffix(final_path.suffix + ".part")

    if file_is_valid(final_path, spec["kind"]):
        print("Valid file already exists; skipping:", final_path.name)
        return final_path

    remote_size = remote_file_size(spec["url"])
    start_byte = part_path.stat().st_size if part_path.exists() else 0
    headers = {"Range": f"bytes={start_byte}-"} if start_byte else {}
    mode = "ab" if start_byte else "wb"

    try:
        with requests.get(spec["url"], stream=True, headers=headers, timeout=(30, 120)) as response:
            response.raise_for_status()
            if start_byte and response.status_code != 206:
                print("Server did not honour resume; restarting this file.")
                start_byte = 0
                mode = "wb"

            total = remote_size or int(response.headers.get("Content-Length", 0)) or None
            initial = start_byte
            with open(part_path, mode) as handle, tqdm(
                total=total,
                initial=initial,
                unit="B",
                unit_scale=True,
                desc=spec["filename"],
            ) as progress:
                for chunk in response.iter_content(chunk_size=chunk_mb * 1024 * 1024):
                    if chunk:
                        handle.write(chunk)
                        progress.update(len(chunk))

        part_path.replace(final_path)
        if not file_is_valid(final_path, spec["kind"]):
            raise RuntimeError("Downloaded file failed its integrity check.")
        return final_path
    except Exception as exc:
        message = (
            f"Automatic download failed for {spec['filename']}.\n"
            f"Download that exact file from the official ISIC Challenge datasets page, then place it here:\n"
            f"{final_path}\n"
            f"Original error: {exc}"
        )
        print(message)
        raise RuntimeError(message) from exc


DOWNLOAD_NOW = True
if DOWNLOAD_NOW:
    dataset_files = {
        key: download_with_resume(spec) for key, spec in DATASET_SPECS.items()
    }
else:
    dataset_files = {
        key: ARCHIVE_DIR / spec["filename"] for key, spec in DATASET_SPECS.items()
    }
    missing = [str(path) for path in dataset_files.values() if not path.exists()]
    if missing:
        raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))


## 6. Safe extraction and disk-space checks


In [ ]:
def zip_uncompressed_size(zip_path):
    with zipfile.ZipFile(zip_path) as archive:
        return sum(info.file_size for info in archive.infolist())


def available_bytes(path):
    return shutil.disk_usage(Path(path)).free


def safe_extract_zip(zip_path, destination, marker_name=".extraction_complete"):
    zip_path = Path(zip_path)
    destination = Path(destination)
    marker = destination / marker_name
    if marker.exists():
        print("Extraction marker found; skipping:", zip_path.name)
        return destination

    required = int(zip_uncompressed_size(zip_path) * 1.15)
    free = available_bytes(destination.parent)
    print(f"{zip_path.name}: approximately {required / 1e9:.1f} GB required; {free / 1e9:.1f} GB free.")
    if free < required:
        raise RuntimeError(
            f"Insufficient runtime disk space to extract {zip_path.name}. "
            f"Free at least {(required - free) / 1e9:.1f} GB, delete unused runtime files, "
            "or change the working directory to a Drive folder before trying again."
        )

    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in tqdm(archive.infolist(), desc=f"Extracting {zip_path.name}"):
            target = (destination / member.filename).resolve()
            if root not in target.parents and target != root:
                raise RuntimeError(f"Unsafe ZIP member path: {member.filename}")
            archive.extract(member, destination)
    marker.write_text(f"Extracted from {zip_path.name}\n", encoding="utf-8")
    return destination


safe_extract_zip(dataset_files["seg_images"], SEG_WORK_DIR / "images")
safe_extract_zip(dataset_files["seg_masks"], SEG_WORK_DIR / "masks")
safe_extract_zip(dataset_files["cls_images"], CLS_WORK_DIR / "images")


## 7. ISIC 2018 segmentation manifest and deterministic splits


In [ ]:
def image_id_from_image_path(path):
    return Path(path).stem


def image_id_from_mask_path(path):
    stem = Path(path).stem
    return re.sub(r"_segmentation$", "", stem, flags=re.IGNORECASE)


seg_image_paths = sorted(
    p for p in (SEG_WORK_DIR / "images").rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg"}
)
seg_mask_paths = sorted(
    p for p in (SEG_WORK_DIR / "masks").rglob("*")
    if p.suffix.lower() == ".png"
)

image_lookup = {image_id_from_image_path(p): str(p) for p in seg_image_paths}
mask_lookup = {image_id_from_mask_path(p): str(p) for p in seg_mask_paths}

if len(image_lookup) != EXPECTED_SEGMENTATION_CASES:
    raise RuntimeError(
        f"Expected {EXPECTED_SEGMENTATION_CASES} unique ISIC 2018 images, found {len(image_lookup)}. "
        "Stop: check that the official Task1-2 training-input ZIP was fully extracted."
    )
if len(mask_lookup) != EXPECTED_SEGMENTATION_CASES:
    raise RuntimeError(
        f"Expected {EXPECTED_SEGMENTATION_CASES} unique masks, found {len(mask_lookup)}. "
        "Stop: check that the official Task 1 ground-truth ZIP was fully extracted."
    )

image_ids = set(image_lookup)
mask_ids = set(mask_lookup)
if image_ids != mask_ids:
    missing_masks = sorted(image_ids - mask_ids)[:10]
    missing_images = sorted(mask_ids - image_ids)[:10]
    raise RuntimeError(
        "Image-mask ID matching failed. "
        f"Example IDs without masks: {missing_masks}; without images: {missing_images}"
    )

seg_manifest = pd.DataFrame({
    "image_id": sorted(image_ids),
    "image_path": [image_lookup[i] for i in sorted(image_ids)],
    "mask_path": [mask_lookup[i] for i in sorted(image_ids)],
})

assert seg_manifest["image_path"].map(Path).map(Path.exists).all()
assert seg_manifest["mask_path"].map(Path).map(Path.exists).all()
seg_manifest.to_csv(MANIFEST_DIR / "segmentation_manifest_full.csv", index=False)

if RUN_MODE == "quick":
    seg_working = seg_manifest.sample(
        n=min(QUICK_SEG_CASES, len(seg_manifest)), random_state=SEED
    ).reset_index(drop=True)
else:
    seg_working = seg_manifest.copy()

train_seg_df, temp_seg_df = train_test_split(
    seg_working, test_size=0.20, random_state=SEED, shuffle=True
)
val_seg_df, test_seg_df = train_test_split(
    temp_seg_df, test_size=0.50, random_state=SEED, shuffle=True
)

for frame in [train_seg_df, val_seg_df, test_seg_df]:
    frame.reset_index(drop=True, inplace=True)

train_ids = set(train_seg_df.image_id)
val_ids = set(val_seg_df.image_id)
test_ids = set(test_seg_df.image_id)
assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)
assert len(train_seg_df) + len(val_seg_df) + len(test_seg_df) == len(seg_working)

train_seg_df.to_csv(MANIFEST_DIR / f"segmentation_{RUN_MODE}_train.csv", index=False)
val_seg_df.to_csv(MANIFEST_DIR / f"segmentation_{RUN_MODE}_validation.csv", index=False)
test_seg_df.to_csv(MANIFEST_DIR / f"segmentation_{RUN_MODE}_test.csv", index=False)

print("Segmentation manifest:", len(seg_manifest))
print("Working split sizes:", {
    "train": len(train_seg_df), "validation": len(val_seg_df), "test": len(test_seg_df)
})


## 8. Segmentation sanity check: image, ground truth, and overlay


In [ ]:
def load_rgb_pil(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()


def load_mask_pil(path):
    with Image.open(path) as mask:
        return mask.convert("L").copy()


def show_segmentation_sample(row):
    image = load_rgb_pil(row["image_path"])
    mask = load_mask_pil(row["mask_path"])
    if image.size != mask.size:
        print("Original dimensions differ; preprocessing will resize them independently:", image.size, mask.size)

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    axes[0].imshow(image)
    axes[0].set_title("Original dermoscopic image")
    axes[1].imshow(mask, cmap="gray")
    axes[1].set_title("Ground-truth mask")
    axes[2].imshow(image)
    axes[2].imshow(mask.resize(image.size, Image.Resampling.NEAREST), cmap="Reds", alpha=0.35)
    axes[2].set_title("Mask overlay")
    for axis in axes:
        axis.axis("off")
    plt.tight_layout()
    plt.show()


show_segmentation_sample(train_seg_df.sample(1, random_state=SEED).iloc[0])


## 9. `tf.data` segmentation pipeline and paired augmentation


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def decode_segmentation_pair(image_path, mask_path):
    image_bytes = tf.io.read_file(image_path)
    image = tf.io.decode_jpeg(image_bytes, channels=3)
    image = tf.image.resize(image, SEG_IMAGE_SIZE, method="bilinear")
    image = tf.cast(image, tf.float32) / 255.0
    image.set_shape((*SEG_IMAGE_SIZE, 3))

    mask_bytes = tf.io.read_file(mask_path)
    mask = tf.io.decode_png(mask_bytes, channels=1)
    mask = tf.image.resize(mask, SEG_IMAGE_SIZE, method="nearest")
    mask = tf.cast(mask > 127, tf.float32)
    mask.set_shape((*SEG_IMAGE_SIZE, 1))
    return image, mask


def _projective_transform(image, transform, interpolation, fill_mode, fill_value=0.0):
    transformed = tf.raw_ops.ImageProjectiveTransformV3(
        images=image[None, ...],
        transforms=transform[None, ...],
        output_shape=tf.cast(tf.shape(image)[:2], tf.int32),
        interpolation=interpolation,
        fill_mode=fill_mode,
        fill_value=tf.cast(fill_value, tf.float32),
    )
    return transformed[0]


def stateless_paired_augmentation(index, pair):
    image, mask = pair
    base_seed = tf.stack([tf.cast(SEED, tf.int32), tf.cast(index % 2147483647, tf.int32)])
    seeds = tf.random.experimental.stateless_split(base_seed, 10)

    horizontal = tf.random.stateless_uniform([], seeds[0]) > 0.5
    vertical = tf.random.stateless_uniform([], seeds[1]) > 0.5
    image = tf.cond(horizontal, lambda: tf.image.flip_left_right(image), lambda: image)
    mask = tf.cond(horizontal, lambda: tf.image.flip_left_right(mask), lambda: mask)
    image = tf.cond(vertical, lambda: tf.image.flip_up_down(image), lambda: image)
    mask = tf.cond(vertical, lambda: tf.image.flip_up_down(mask), lambda: mask)

    angle = tf.random.stateless_uniform([], seeds[2], -10.0, 10.0) * (math.pi / 180.0)
    zoom = tf.random.stateless_uniform([], seeds[3], 0.92, 1.08)
    tx = tf.random.stateless_uniform([], seeds[4], -0.05, 0.05) * tf.cast(SEG_IMAGE_SIZE[1], tf.float32)
    ty = tf.random.stateless_uniform([], seeds[5], -0.05, 0.05) * tf.cast(SEG_IMAGE_SIZE[0], tf.float32)

    height = tf.cast(SEG_IMAGE_SIZE[0], tf.float32)
    width = tf.cast(SEG_IMAGE_SIZE[1], tf.float32)
    cx = (width - 1.0) / 2.0
    cy = (height - 1.0) / 2.0
    cos_a = tf.cos(angle) / zoom
    sin_a = tf.sin(angle) / zoom
    a0, a1 = cos_a, sin_a
    b0, b1 = -sin_a, cos_a
    a2 = cx - a0 * (cx + tx) - a1 * (cy + ty)
    b2 = cy - b0 * (cx + tx) - b1 * (cy + ty)
    transform = tf.stack([a0, a1, a2, b0, b1, b2, 0.0, 0.0])

    image = _projective_transform(image, transform, "BILINEAR", "REFLECT")
    mask = _projective_transform(mask, transform, "NEAREST", "CONSTANT", 0.0)

    image = tf.image.stateless_random_brightness(image, max_delta=0.06, seed=seeds[6])
    image = tf.image.stateless_random_contrast(image, lower=0.90, upper=1.10, seed=seeds[7])
    image = tf.clip_by_value(image, 0.0, 1.0)
    mask = tf.cast(mask >= 0.5, tf.float32)
    return image, mask


def make_segmentation_dataset(frame, training=False):
    ds = tf.data.Dataset.from_tensor_slices((
        frame["image_path"].astype(str).values,
        frame["mask_path"].astype(str).values,
    ))
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_segmentation_pair, num_parallel_calls=AUTOTUNE, deterministic=True)
    if training:
        ds = ds.enumerate()
        ds = ds.map(stateless_paired_augmentation, num_parallel_calls=AUTOTUNE, deterministic=True)
    ds = ds.batch(SEG_BATCH_SIZE, drop_remainder=False)
    return ds.prefetch(AUTOTUNE)


train_seg_ds = make_segmentation_dataset(train_seg_df, training=True)
val_seg_ds = make_segmentation_dataset(val_seg_df, training=False)
test_seg_ds = make_segmentation_dataset(test_seg_df, training=False)

aug_images, aug_masks = next(iter(train_seg_ds))
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(aug_images[0].numpy())
axes[0].set_title("Augmented image")
axes[1].imshow(aug_masks[0, ..., 0].numpy(), cmap="gray")
axes[1].set_title("Identically transformed mask")
axes[2].imshow(aug_images[0].numpy())
axes[2].imshow(aug_masks[0, ..., 0].numpy(), cmap="Reds", alpha=0.35)
axes[2].set_title("Alignment check")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()


## 10. Segmentation metrics and BCE + Dice loss


In [ ]:
SMOOTH = tf.keras.backend.epsilon()


@keras.utils.register_keras_serializable(package="MelanomaAI")
def dice_coefficient(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    axes = tuple(range(1, len(y_true.shape)))
    intersection = tf.reduce_sum(y_true * y_pred, axis=axes)
    denominator = tf.reduce_sum(y_true, axis=axes) + tf.reduce_sum(y_pred, axis=axes)
    return tf.reduce_mean((2.0 * intersection + SMOOTH) / (denominator + SMOOTH))


@keras.utils.register_keras_serializable(package="MelanomaAI")
def iou_coefficient(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    axes = tuple(range(1, len(y_true.shape)))
    intersection = tf.reduce_sum(y_true * y_pred, axis=axes)
    union = tf.reduce_sum(y_true + y_pred - y_true * y_pred, axis=axes)
    return tf.reduce_mean((intersection + SMOOTH) / (union + SMOOTH))


@keras.utils.register_keras_serializable(package="MelanomaAI")
def pixel_sensitivity(y_true, y_pred):
    y_true = tf.cast(y_true >= 0.5, tf.float32)
    y_pred = tf.cast(y_pred >= 0.5, tf.float32)
    tp = tf.reduce_sum(y_true * y_pred)
    fn = tf.reduce_sum(y_true * (1.0 - y_pred))
    return (tp + SMOOTH) / (tp + fn + SMOOTH)


@keras.utils.register_keras_serializable(package="MelanomaAI")
def pixel_specificity(y_true, y_pred):
    y_true = tf.cast(y_true >= 0.5, tf.float32)
    y_pred = tf.cast(y_pred >= 0.5, tf.float32)
    tn = tf.reduce_sum((1.0 - y_true) * (1.0 - y_pred))
    fp = tf.reduce_sum((1.0 - y_true) * y_pred)
    return (tn + SMOOTH) / (tn + fp + SMOOTH)


@keras.utils.register_keras_serializable(package="MelanomaAI")
def dice_loss(y_true, y_pred):
    return 1.0 - dice_coefficient(y_true, y_pred)


@keras.utils.register_keras_serializable(package="MelanomaAI")
def combined_segmentation_loss(y_true, y_pred):
    bce = tf.reduce_mean(keras.losses.binary_crossentropy(y_true, y_pred))
    return bce + dice_loss(y_true, y_pred)


# Synthetic metric checks, including empty masks. These must remain finite.
metric_cases = [
    (tf.zeros((2, 16, 16, 1)), tf.zeros((2, 16, 16, 1))),
    (tf.ones((2, 16, 16, 1)), tf.ones((2, 16, 16, 1))),
    (tf.ones((2, 16, 16, 1)), tf.zeros((2, 16, 16, 1))),
]
for true_mask, pred_mask in metric_cases:
    values = [dice_coefficient(true_mask, pred_mask), iou_coefficient(true_mask, pred_mask)]
    assert all(np.isfinite(float(v.numpy())) for v in values)
print("Dice/IoU synthetic checks passed.")


## 11. Standard U-Net and Attention U-Net


In [ ]:
def convolution_block(x, filters, name):
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False, name=f"{name}_conv1")(x)
    x = layers.BatchNormalization(name=f"{name}_bn1")(x)
    x = layers.Activation("relu", name=f"{name}_relu1")(x)
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False, name=f"{name}_conv2")(x)
    x = layers.BatchNormalization(name=f"{name}_bn2")(x)
    return layers.Activation("relu", name=f"{name}_relu2")(x)


def attention_gate(skip, gate, intermediate_filters, name):
    theta = layers.Conv2D(intermediate_filters, 1, padding="same", name=f"{name}_theta")(skip)
    phi = layers.Conv2D(intermediate_filters, 1, padding="same", name=f"{name}_phi")(gate)
    merged = layers.Add(name=f"{name}_add")([theta, phi])
    merged = layers.Activation("relu", name=f"{name}_relu")(merged)
    coefficients = layers.Conv2D(1, 1, padding="same", name=f"{name}_psi")(merged)
    coefficients = layers.Activation("sigmoid", dtype="float32", name=f"{name}_coefficients")(coefficients)
    return layers.Multiply(name=f"{name}_weighted_skip")([skip, coefficients])


def build_unet(input_shape=(*SEG_IMAGE_SIZE, 3), attention=False, model_name="unet"):
    inputs = keras.Input(shape=input_shape, name="dermoscopic_image")
    filters = [32, 64, 128, 256]
    skips = []
    x = inputs

    for level, n_filters in enumerate(filters, start=1):
        x = convolution_block(x, n_filters, f"encoder_{level}")
        skips.append(x)
        x = layers.MaxPooling2D(pool_size=2, name=f"pool_{level}")(x)

    x = convolution_block(x, 512, "bottleneck")

    for level, (n_filters, skip) in enumerate(zip(reversed(filters), reversed(skips)), start=1):
        x = layers.Conv2DTranspose(
            n_filters, 2, strides=2, padding="same", name=f"decoder_{level}_upsample"
        )(x)
        if attention:
            skip = attention_gate(skip, x, max(n_filters // 2, 1), f"attention_{level}")
        x = layers.Concatenate(name=f"decoder_{level}_concat")([x, skip])
        x = convolution_block(x, n_filters, f"decoder_{level}")

    outputs = layers.Conv2D(
        1, 1, activation="sigmoid", dtype="float32", name="lesion_probability"
    )(x)
    return keras.Model(inputs, outputs, name=model_name)


def build_standard_unet(input_shape=(*SEG_IMAGE_SIZE, 3)):
    return build_unet(input_shape=input_shape, attention=False, model_name="standard_unet")


def build_attention_unet(input_shape=(*SEG_IMAGE_SIZE, 3)):
    return build_unet(input_shape=input_shape, attention=True, model_name="attention_unet")


standard_unet_preview = build_standard_unet()
attention_unet_preview = build_attention_unet()
print("Standard U-Net parameters:", f"{standard_unet_preview.count_params():,}")
print("Attention U-Net parameters:", f"{attention_unet_preview.count_params():,}")

synthetic_seg_input = tf.zeros((1, *SEG_IMAGE_SIZE, 3), dtype=tf.float32)
assert standard_unet_preview(synthetic_seg_input, training=False).shape == (1, *SEG_IMAGE_SIZE, 1)
assert attention_unet_preview(synthetic_seg_input, training=False).shape == (1, *SEG_IMAGE_SIZE, 1)
print("Both U-Net synthetic forward passes succeeded.")

del standard_unet_preview, attention_unet_preview, synthetic_seg_input
gc.collect()


### Architecture notes

- The two models use the same input size, encoder depth, filter schedule, bottleneck, decoder schedule, loss, optimizer, augmentation, callbacks, and splits.
- The deliberate difference is the attention gating of encoder skip features in Attention U-Net.
- `BatchNormalization` supports stable optimisation, while the final sigmoid layer is forced to `float32` so mixed precision does not reduce output precision.


## 12. Fair segmentation training and callbacks


In [ ]:
SEGMENTATION_MODEL_PATHS = {
    "Standard U-Net": MODELS_DIR / "standard_unet_best.keras",
    "Attention U-Net": MODELS_DIR / "attention_unet_best.keras",
}


def compile_segmenter(model):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=SEG_LEARNING_RATE),
        loss=combined_segmentation_loss,
        metrics=[dice_coefficient, iou_coefficient, pixel_sensitivity, pixel_specificity],
    )
    return model


def segmentation_callbacks(model_path):
    return [
        keras.callbacks.ModelCheckpoint(
            model_path,
            monitor="val_dice_coefficient",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_dice_coefficient",
            mode="max",
            patience=10,
            restore_best_weights=False,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.5,
            patience=4,
            min_lr=1e-6,
            verbose=1,
        ),
    ]


def train_one_segmenter(display_name, builder):
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)
    model = compile_segmenter(builder())
    print(f"\n{display_name}: {model.count_params():,} parameters")

    # Rebuild the same deterministic data definitions for each model.
    training_ds = make_segmentation_dataset(train_seg_df, training=True)
    validation_ds = make_segmentation_dataset(val_seg_df, training=False)
    started = time.perf_counter()
    history = model.fit(
        training_ds,
        validation_data=validation_ds,
        epochs=SEG_EPOCHS,
        callbacks=segmentation_callbacks(SEGMENTATION_MODEL_PATHS[display_name]),
        verbose=1,
    )
    elapsed = time.perf_counter() - started
    history_df = pd.DataFrame(history.history)
    history_df.index.name = "epoch"
    history_df.to_csv(
        RESULTS_DIR / f"{display_name.lower().replace(' ', '_').replace('-', '')}_history.csv"
    )
    record = {
        "display_name": display_name,
        "model_path": str(SEGMENTATION_MODEL_PATHS[display_name]),
        "parameters": int(model.count_params()),
        "training_time_seconds": float(elapsed),
        "history": history_df,
    }
    del model, training_ds, validation_ds
    gc.collect()
    return record


segmentation_runs = {
    "Standard U-Net": train_one_segmenter("Standard U-Net", build_standard_unet),
    "Attention U-Net": train_one_segmenter("Attention U-Net", build_attention_unet),
}


## 13. Validation comparison, locked selection, and held-out test evaluation


In [ ]:
CUSTOM_OBJECTS = {
    "dice_coefficient": dice_coefficient,
    "iou_coefficient": iou_coefficient,
    "pixel_sensitivity": pixel_sensitivity,
    "pixel_specificity": pixel_specificity,
    "dice_loss": dice_loss,
    "combined_segmentation_loss": combined_segmentation_loss,
}


def evaluate_segmenter_streaming(model, dataset, threshold=0.5):
    dice_values, iou_values = [], []
    tp = tn = fp = fn = 0.0
    for images, true_masks in tqdm(dataset, desc="Evaluating segmentation", leave=False):
        probabilities = model.predict_on_batch(images).astype(np.float32)
        predicted = probabilities >= threshold
        truth = true_masks.numpy() >= 0.5
        axes = (1, 2, 3)
        intersection = np.sum(predicted & truth, axis=axes)
        pred_sum = np.sum(predicted, axis=axes)
        true_sum = np.sum(truth, axis=axes)
        union = np.sum(predicted | truth, axis=axes)
        dice_values.extend(((2 * intersection + 1e-7) / (pred_sum + true_sum + 1e-7)).tolist())
        iou_values.extend(((intersection + 1e-7) / (union + 1e-7)).tolist())
        tp += np.sum(predicted & truth)
        tn += np.sum((~predicted) & (~truth))
        fp += np.sum(predicted & (~truth))
        fn += np.sum((~predicted) & truth)
    return {
        "dice": float(np.mean(dice_values)),
        "iou": float(np.mean(iou_values)),
        "pixel_sensitivity": float((tp + 1e-7) / (tp + fn + 1e-7)),
        "pixel_specificity": float((tn + 1e-7) / (tn + fp + 1e-7)),
    }


comparison_rows = []
for display_name, run in segmentation_runs.items():
    model = keras.models.load_model(run["model_path"], custom_objects=CUSTOM_OBJECTS, compile=False)
    validation_metrics = evaluate_segmenter_streaming(model, val_seg_ds, SEG_THRESHOLD)
    comparison_rows.append({
        "model": display_name,
        "validation_dice": validation_metrics["dice"],
        "validation_iou": validation_metrics["iou"],
        "validation_pixel_sensitivity": validation_metrics["pixel_sensitivity"],
        "validation_pixel_specificity": validation_metrics["pixel_specificity"],
        "parameters": run["parameters"],
        "training_time_seconds": run["training_time_seconds"],
        "test_dice": np.nan,
        "test_iou": np.nan,
        "test_pixel_sensitivity": np.nan,
        "test_pixel_specificity": np.nan,
    })
    del model
    gc.collect()

segmentation_comparison = pd.DataFrame(comparison_rows)
segmentation_comparison = segmentation_comparison.sort_values(
    ["validation_dice", "validation_iou"], ascending=False
).reset_index(drop=True)

# Lock the model choice before any test metric is calculated.
SELECTED_SEGMENTER_NAME = segmentation_comparison.loc[0, "model"]
SELECTED_SEGMENTER_SOURCE = Path(
    segmentation_runs[SELECTED_SEGMENTER_NAME]["model_path"]
)
SELECTED_SEGMENTER_PATH = MODELS_DIR / "selected_segmenter.keras"
shutil.copy2(SELECTED_SEGMENTER_SOURCE, SELECTED_SEGMENTER_PATH)
print("Selected from validation results only:", SELECTED_SEGMENTER_NAME)

# Final, one-time post-selection audit of both candidates on the untouched test split.
# These test numbers are descriptive and cannot change the locked model choice.
for row_index, row in segmentation_comparison.iterrows():
    model_path = segmentation_runs[row["model"]]["model_path"]
    model = keras.models.load_model(model_path, custom_objects=CUSTOM_OBJECTS, compile=False)
    metrics = evaluate_segmenter_streaming(model, test_seg_ds, SEG_THRESHOLD)
    for key, value in metrics.items():
        column = "test_" + key
        segmentation_comparison.loc[row_index, column] = value
    del model
    gc.collect()

segmentation_comparison.to_csv(RESULTS_DIR / "segmentation_results.csv", index=False)
(RESULTS_DIR / "segmentation_results.json").write_text(
    segmentation_comparison.to_json(orient="records", indent=2), encoding="utf-8"
)
display(segmentation_comparison)

selected_segmenter = keras.models.load_model(
    SELECTED_SEGMENTER_PATH, custom_objects=CUSTOM_OBJECTS, compile=False
)


## 14. Segmentation learning curves and qualitative test results


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for display_name, run in segmentation_runs.items():
    history = run["history"]
    epochs = np.arange(1, len(history) + 1)
    axes[0, 0].plot(epochs, history["dice_coefficient"], label=display_name)
    axes[0, 1].plot(epochs, history["val_dice_coefficient"], label=display_name)
    axes[1, 0].plot(epochs, history["loss"], label=display_name)
    axes[1, 1].plot(epochs, history["val_loss"], label=display_name)

titles = ["Training Dice", "Validation Dice", "Training loss", "Validation loss"]
for axis, title in zip(axes.flat, titles):
    axis.set_title(title)
    axis.set_xlabel("Epoch")
    axis.grid(alpha=0.25)
    axis.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "segmentation_learning_curves.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
def predict_resized_segmentation(model, image_pil):
    resized = image_pil.resize((SEG_IMAGE_SIZE[1], SEG_IMAGE_SIZE[0]), Image.Resampling.BILINEAR)
    array = np.asarray(resized, dtype=np.float32) / 255.0
    probability = model.predict(array[None, ...], verbose=0)[0, ..., 0]
    return probability


qualitative_rows = test_seg_df.sample(
    n=min(6, len(test_seg_df)), random_state=SEED
).reset_index(drop=True)

fig, axes = plt.subplots(len(qualitative_rows), 4, figsize=(14, 3.5 * len(qualitative_rows)))
axes = np.atleast_2d(axes)
for row_number, (_, row) in enumerate(qualitative_rows.iterrows()):
    image = load_rgb_pil(row.image_path)
    true_mask = load_mask_pil(row.mask_path).resize(
        (SEG_IMAGE_SIZE[1], SEG_IMAGE_SIZE[0]), Image.Resampling.NEAREST
    )
    probability = predict_resized_segmentation(selected_segmenter, image)
    predicted = probability >= SEG_THRESHOLD
    resized_image = image.resize((SEG_IMAGE_SIZE[1], SEG_IMAGE_SIZE[0]), Image.Resampling.BILINEAR)

    axes[row_number, 0].imshow(resized_image)
    axes[row_number, 0].set_title("Image")
    axes[row_number, 1].imshow(true_mask, cmap="gray")
    axes[row_number, 1].set_title("Ground truth")
    axes[row_number, 2].imshow(predicted, cmap="gray")
    axes[row_number, 2].set_title("Predicted mask")
    axes[row_number, 3].imshow(resized_image)
    axes[row_number, 3].imshow(predicted, cmap="Reds", alpha=0.35)
    axes[row_number, 3].set_title("Prediction overlay")
    for axis in axes[row_number]:
        axis.axis("off")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "selected_segmenter_test_examples.png", dpi=180, bbox_inches="tight")
plt.show()


## 15. Mask postprocessing and safe padded lesion ROI extraction


In [ ]:
def postprocess_binary_mask(probability_mask, threshold=SEG_THRESHOLD):
    mask = np.asarray(probability_mask, dtype=np.float32) >= float(threshold)
    if not np.any(mask):
        return mask.astype(np.uint8)
    labels, count = ndimage.label(mask)
    if count > 1:
        sizes = ndimage.sum(mask, labels, index=np.arange(1, count + 1))
        largest_label = int(np.argmax(sizes)) + 1
        mask = labels == largest_label
    mask = ndimage.binary_fill_holes(mask)
    return mask.astype(np.uint8)


def extract_lesion_roi(
    image,
    predicted_mask,
    padding_fraction=ROI_PADDING_FRACTION,
    min_fraction=MIN_VALID_MASK_FRACTION,
    max_fraction=MAX_VALID_MASK_FRACTION,
):
    image_array = np.asarray(image.convert("RGB") if isinstance(image, Image.Image) else image)
    if image_array.ndim != 3 or image_array.shape[2] != 3:
        raise ValueError("image must be RGB with shape H×W×3")

    mask = np.asarray(predicted_mask).squeeze().astype(bool)
    height, width = image_array.shape[:2]
    if mask.shape != (height, width):
        mask = np.asarray(
            Image.fromarray(mask.astype(np.uint8) * 255).resize(
                (width, height), Image.Resampling.NEAREST
            )
        ) > 127

    fraction = float(mask.mean())
    fallback_reason = None
    if not np.any(mask):
        fallback_reason = "empty_mask"
    elif fraction < min_fraction:
        fallback_reason = "mask_too_small"
    elif fraction > max_fraction:
        fallback_reason = "mask_too_large"

    if fallback_reason is not None:
        return image_array.copy(), (0, 0, width, height), True, fallback_reason

    ys, xs = np.where(mask)
    x0, x1 = int(xs.min()), int(xs.max()) + 1
    y0, y1 = int(ys.min()), int(ys.max()) + 1
    pad_x = max(1, int((x1 - x0) * padding_fraction))
    pad_y = max(1, int((y1 - y0) * padding_fraction))
    x0, x1 = max(0, x0 - pad_x), min(width, x1 + pad_x)
    y0, y1 = max(0, y0 - pad_y), min(height, y1 + pad_y)

    if x1 <= x0 or y1 <= y0:
        return image_array.copy(), (0, 0, width, height), True, "invalid_bounding_box"
    return image_array[y0:y1, x0:x1].copy(), (x0, y0, x1, y1), False, None


# Required synthetic ROI tests: normal, empty, and full-image masks.
synthetic_image = Image.fromarray(np.full((100, 120, 3), 128, dtype=np.uint8))
normal_mask = np.zeros((100, 120), dtype=np.uint8)
normal_mask[30:70, 40:90] = 1
empty_mask = np.zeros((100, 120), dtype=np.uint8)
full_mask = np.ones((100, 120), dtype=np.uint8)

normal_result = extract_lesion_roi(synthetic_image, normal_mask)
empty_result = extract_lesion_roi(synthetic_image, empty_mask)
full_result = extract_lesion_roi(synthetic_image, full_mask)
assert normal_result[2] is False
assert empty_result[2] is True and empty_result[3] == "empty_mask"
assert full_result[2] is True and full_result[3] == "mask_too_large"
print("ROI synthetic checks passed.")


## 16. SIIM-ISIC 2020 classification manifest, duplicate removal, and patient-level splitting


In [ ]:
ground_truth = pd.read_csv(dataset_files["cls_ground_truth"])
metadata_v2 = pd.read_csv(dataset_files["cls_metadata_v2"])
duplicate_pairs = pd.read_csv(dataset_files["cls_duplicates"])

required_gt = {"image_name", "target"}
required_meta = {"image_name", "patient_id"}
if not required_gt.issubset(ground_truth.columns):
    raise RuntimeError(f"Ground-truth columns are {ground_truth.columns.tolist()}, expected at least {sorted(required_gt)}")
if not required_meta.issubset(metadata_v2.columns):
    raise RuntimeError(f"Metadata-v2 columns are {metadata_v2.columns.tolist()}, expected at least {sorted(required_meta)}")

if ground_truth["image_name"].duplicated().any() or metadata_v2["image_name"].duplicated().any():
    raise RuntimeError("Unexpected duplicate image_name rows in an official ground-truth/metadata file.")

metadata_columns = [
    column for column in [
        "image_name", "patient_id", "lesion_id", "sex", "age_approx",
        "anatom_site_general_challenge", "diagnosis", "benign_malignant"
    ] if column in metadata_v2.columns
]
classification_manifest = metadata_v2[metadata_columns].copy()
classification_manifest = classification_manifest.merge(
    ground_truth[["image_name", "target"]], on="image_name", how="inner", validate="one_to_one"
)

if len(classification_manifest) != EXPECTED_CLASSIFICATION_IMAGES:
    raise RuntimeError(
        f"Expected {EXPECTED_CLASSIFICATION_IMAGES} joined labelled records, found {len(classification_manifest)}. "
        "Stop: inspect the two official 2020 CSV files."
    )

classification_manifest["target"] = pd.to_numeric(classification_manifest["target"], errors="coerce")
if classification_manifest["target"].isna().any():
    raise RuntimeError("NaN/non-numeric values were found in the required target column.")
classification_manifest["target"] = classification_manifest["target"].astype(int)
if not set(classification_manifest["target"].unique()).issubset({0, 1}):
    raise RuntimeError("The supplied target must be binary 0/1.")
if classification_manifest["patient_id"].isna().any():
    raise RuntimeError("Patient-level splitting is impossible because patient_id contains NaNs.")
classification_manifest["patient_id"] = classification_manifest["patient_id"].astype(str)

cls_image_paths = [
    p for p in (CLS_WORK_DIR / "images").rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg"}
]
cls_image_lookup = {p.stem: str(p) for p in cls_image_paths}
if len(cls_image_lookup) != EXPECTED_CLASSIFICATION_IMAGES:
    raise RuntimeError(
        f"Expected {EXPECTED_CLASSIFICATION_IMAGES} unique JPEG images, found {len(cls_image_lookup)}. "
        "Stop: check the official 2020 JPEG ZIP extraction."
    )
classification_manifest["original_path"] = classification_manifest["image_name"].map(cls_image_lookup)
if classification_manifest["original_path"].isna().any():
    missing = classification_manifest.loc[
        classification_manifest["original_path"].isna(), "image_name"
    ].head(10).tolist()
    raise RuntimeError(f"Some labelled images were not found. Examples: {missing}")

if not {"image_name_1", "image_name_2"}.issubset(duplicate_pairs.columns):
    raise RuntimeError(
        "Official duplicate CSV did not contain image_name_1 and image_name_2; "
        f"found {duplicate_pairs.columns.tolist()}. Stop rather than guessing."
    )
if len(duplicate_pairs) != EXPECTED_DUPLICATE_PAIRS:
    raise RuntimeError(
        f"Expected {EXPECTED_DUPLICATE_PAIRS} official duplicate pairs, found {len(duplicate_pairs)}."
    )

# Retain image_name_1 as the representative and remove image_name_2 from each identical pair.
duplicate_ids_to_remove = set(duplicate_pairs["image_name_2"].astype(str))
missing_duplicate_ids = duplicate_ids_to_remove - set(classification_manifest.image_name)
if missing_duplicate_ids:
    raise RuntimeError(f"Duplicate-list image IDs were absent from the manifest: {sorted(missing_duplicate_ids)[:10]}")

classification_clean = classification_manifest.loc[
    ~classification_manifest["image_name"].isin(duplicate_ids_to_remove)
].reset_index(drop=True)
assert len(classification_manifest) - len(classification_clean) == EXPECTED_DUPLICATE_PAIRS

classification_manifest.to_csv(MANIFEST_DIR / "classification_manifest_before_duplicate_removal.csv", index=False)
classification_clean.to_csv(MANIFEST_DIR / "classification_manifest_duplicate_clean.csv", index=False)
duplicate_pairs.to_csv(MANIFEST_DIR / "official_duplicate_pairs_used.csv", index=False)

print("Before duplicate removal:", len(classification_manifest))
print("Removed duplicate copies:", len(duplicate_ids_to_remove))
print("After duplicate removal:", len(classification_clean))


In [ ]:
def stratified_patient_subset(frame, max_images, seed=SEED):
    if len(frame) <= max_images:
        return frame.copy().reset_index(drop=True)
    patient_summary = frame.groupby("patient_id", as_index=False).agg(
        target=("target", "max"),
        n_images=("image_name", "size"),
    )
    average_images = max(float(patient_summary.n_images.mean()), 1.0)
    n_patients = max(20, int(math.ceil(max_images / average_images)))
    n_patients = min(n_patients, len(patient_summary) - 1)
    sampled_patients, _ = train_test_split(
        patient_summary,
        train_size=n_patients,
        stratify=patient_summary["target"],
        random_state=seed,
    )
    subset = frame[frame.patient_id.isin(sampled_patients.patient_id)].copy()
    print(f"Quick subset selected {len(sampled_patients)} whole patients and {len(subset)} images.")
    return subset.reset_index(drop=True)


if RUN_MODE == "quick":
    cls_working = stratified_patient_subset(classification_clean, QUICK_CLS_MAX_IMAGES)
else:
    cls_working = classification_clean.copy()


def choose_best_group_holdout(frame, n_splits, desired_fraction, seed):
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    global_prevalence = frame.target.mean()
    best = None
    for fold_number, (keep_idx, holdout_idx) in enumerate(
        splitter.split(frame, frame.target, groups=frame.patient_id)
    ):
        holdout = frame.iloc[holdout_idx]
        size_difference = abs(len(holdout) / len(frame) - desired_fraction)
        prevalence_difference = abs(holdout.target.mean() - global_prevalence)
        score = size_difference + prevalence_difference
        candidate = (score, fold_number, keep_idx, holdout_idx)
        if best is None or candidate[0] < best[0]:
            best = candidate
    return best[2], best[3]


trainval_idx, test_idx = choose_best_group_holdout(
    cls_working, n_splits=10, desired_fraction=0.10, seed=SEED
)
cls_trainval = cls_working.iloc[trainval_idx].reset_index(drop=True)
cls_test_df = cls_working.iloc[test_idx].reset_index(drop=True)

train_idx, val_idx = choose_best_group_holdout(
    cls_trainval, n_splits=9, desired_fraction=1.0 / 9.0, seed=SEED + 1
)
cls_train_df = cls_trainval.iloc[train_idx].reset_index(drop=True)
cls_val_df = cls_trainval.iloc[val_idx].reset_index(drop=True)

train_patients = set(cls_train_df.patient_id)
val_patients = set(cls_val_df.patient_id)
test_patients = set(cls_test_df.patient_id)
assert train_patients.isdisjoint(val_patients)
assert train_patients.isdisjoint(test_patients)
assert val_patients.isdisjoint(test_patients)
assert len(cls_train_df) + len(cls_val_df) + len(cls_test_df) == len(cls_working)
for split_name, split_frame in [
    ("train", cls_train_df), ("validation", cls_val_df), ("test", cls_test_df)
]:
    if set(split_frame.target.unique()) != {0, 1}:
        raise RuntimeError(
            f"The {split_name} patient-level split does not contain both classes. "
            "Increase the quick-mode subset or use full mode."
        )


def classification_split_summary(name, frame):
    benign = int((frame.target == 0).sum())
    malignant = int((frame.target == 1).sum())
    result = {
        "split": name,
        "images": len(frame),
        "patients": frame.patient_id.nunique(),
        "benign": benign,
        "malignant": malignant,
        "benign_percent": 100.0 * benign / len(frame),
        "malignant_percent": 100.0 * malignant / len(frame),
    }
    print(result)
    return result


split_summary = pd.DataFrame([
    classification_split_summary("train", cls_train_df),
    classification_split_summary("validation", cls_val_df),
    classification_split_summary("test", cls_test_df),
])
split_summary.to_csv(RESULTS_DIR / "classification_split_summary.csv", index=False)

for name, frame in [
    ("train", cls_train_df), ("validation", cls_val_df), ("test", cls_test_df)
]:
    frame.to_csv(MANIFEST_DIR / f"classification_{RUN_MODE}_{name}_pre_roi.csv", index=False)

print("ZERO patient overlap confirmed across train, validation, and test.")
display(split_summary)


## 17. Corrupt-file check and batched ROI generation


In [ ]:
def find_unreadable_images(frame, path_column):
    unreadable = []
    for row in tqdm(frame[["image_name", path_column]].itertuples(index=False),
                    total=len(frame), desc="Checking image files"):
        image_name, path = row
        try:
            with Image.open(path) as image:
                image.verify()
        except (OSError, ValueError, UnidentifiedImageError) as exc:
            unreadable.append({"image_name": image_name, "path": path, "error": repr(exc)})
    return pd.DataFrame(unreadable, columns=["image_name", "path", "error"])


if VERIFY_IMAGE_CONTENT:
    corrupt_df = find_unreadable_images(cls_working, "original_path")
else:
    corrupt_df = pd.DataFrame(columns=["image_name", "path", "error"])
    print("Content verification disabled; file existence was still asserted.")

corrupt_df.to_csv(RESULTS_DIR / f"corrupt_classification_images_{RUN_MODE}.csv", index=False)
if len(corrupt_df):
    print(f"Removing and recording {len(corrupt_df)} unreadable images rather than crashing later.")
    corrupt_ids = set(corrupt_df.image_name)
    cls_train_df = cls_train_df[~cls_train_df.image_name.isin(corrupt_ids)].reset_index(drop=True)
    cls_val_df = cls_val_df[~cls_val_df.image_name.isin(corrupt_ids)].reset_index(drop=True)
    cls_test_df = cls_test_df[~cls_test_df.image_name.isin(corrupt_ids)].reset_index(drop=True)
else:
    print("No unreadable images found in the working classification set.")


In [ ]:
def decode_segmentation_inference(path):
    image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    image = tf.image.resize(image, SEG_IMAGE_SIZE, method="bilinear")
    image = tf.cast(image, tf.float32) / 255.0
    image.set_shape((*SEG_IMAGE_SIZE, 3))
    return path, image


def make_segmentation_inference_dataset(paths):
    ds = tf.data.Dataset.from_tensor_slices(np.asarray(paths, dtype=str))
    ds = ds.map(decode_segmentation_inference, num_parallel_calls=AUTOTUNE, deterministic=True)
    return ds.batch(SEG_BATCH_SIZE).prefetch(AUTOTUNE)


def generate_roi_manifest(frame, segmenter, checkpoint_path):
    frame = frame.copy().reset_index(drop=True)
    checkpoint_path = Path(checkpoint_path)
    completed = {}
    if checkpoint_path.exists():
        prior = pd.read_csv(checkpoint_path)
        for row in prior.to_dict("records"):
            if row.get("roi_path") and Path(str(row["roi_path"])).exists():
                completed[str(row["image_name"])] = row

    pending = frame[~frame.image_name.astype(str).isin(completed)].copy()
    pending_lookup = pending.set_index("original_path").to_dict(orient="index")
    print(f"ROI generation: {len(completed)} reusable; {len(pending)} pending.")
    new_records = []
    inference_ds = make_segmentation_inference_dataset(pending.original_path.astype(str).tolist())
    processed = 0

    for path_batch, image_batch in tqdm(inference_ds, desc="Segmenting 2020 images"):
        probabilities = segmenter.predict_on_batch(image_batch).astype(np.float32)[..., 0]
        for raw_path, probability in zip(path_batch.numpy(), probabilities):
            original_path = raw_path.decode("utf-8")
            source_row = pending_lookup[original_path]
            try:
                original = load_rgb_pil(original_path)
                probability_original = np.asarray(
                    Image.fromarray(probability.astype(np.float32), mode="F").resize(
                        original.size, Image.Resampling.BILINEAR
                    ), dtype=np.float32
                )
                binary_mask = postprocess_binary_mask(probability_original, SEG_THRESHOLD)
                roi, bbox, fallback, reason = extract_lesion_roi(original, binary_mask)
                roi_path = ROI_DIR / f"{source_row['image_name']}.jpg"
                Image.fromarray(roi).save(roi_path, format="JPEG", quality=95, optimize=True)
                record = dict(source_row)
                record["original_path"] = original_path
                record.update({
                    "roi_path": str(roi_path),
                    "bounding_box": json.dumps([int(v) for v in bbox]),
                    "roi_fallback_used": bool(fallback),
                    "roi_fallback_reason": reason,
                })
                new_records.append(record)
            except Exception as exc:
                warnings.warn(f"ROI generation failed for {source_row['image_name']}: {exc}; using full image.")
                original = load_rgb_pil(original_path)
                roi_path = ROI_DIR / f"{source_row['image_name']}.jpg"
                original.save(roi_path, format="JPEG", quality=95, optimize=True)
                record = dict(source_row)
                record["original_path"] = original_path
                record.update({
                    "roi_path": str(roi_path),
                    "bounding_box": json.dumps([0, 0, original.width, original.height]),
                    "roi_fallback_used": True,
                    "roi_fallback_reason": f"processing_error:{type(exc).__name__}",
                })
                new_records.append(record)

        processed += len(path_batch)
        if processed % 500 < SEG_BATCH_SIZE:
            checkpoint_records = list(completed.values()) + new_records
            pd.DataFrame(checkpoint_records).to_csv(checkpoint_path, index=False)

    all_records = list(completed.values()) + new_records
    result = pd.DataFrame(all_records)
    if len(result) != len(frame) or result.image_name.nunique() != len(frame):
        raise RuntimeError("ROI manifest is incomplete or has duplicate image names.")
    if not result.roi_path.map(Path).map(Path.exists).all():
        raise RuntimeError("One or more ROI files were not saved successfully.")
    result = frame[["image_name"]].merge(result, on="image_name", how="left", validate="one_to_one")
    result.to_csv(checkpoint_path, index=False)
    return result


classification_for_roi = pd.concat(
    [cls_train_df, cls_val_df, cls_test_df], ignore_index=True
).drop_duplicates("image_name")
roi_manifest_path = MANIFEST_DIR / f"classification_roi_manifest_{RUN_MODE}.csv"
classification_roi_manifest = generate_roi_manifest(
    classification_for_roi, selected_segmenter, roi_manifest_path
)

roi_columns = [
    "image_name", "patient_id", "lesion_id", "original_path", "roi_path",
    "target", "bounding_box", "roi_fallback_used", "roi_fallback_reason"
]
roi_columns = [column for column in roi_columns if column in classification_roi_manifest.columns]
classification_roi_manifest[roi_columns].to_csv(roi_manifest_path, index=False)


def attach_roi_paths(split_frame):
    roi_only = classification_roi_manifest[[
        "image_name", "roi_path", "bounding_box", "roi_fallback_used", "roi_fallback_reason"
    ]]
    return split_frame.merge(roi_only, on="image_name", how="left", validate="one_to_one")


cls_train_df = attach_roi_paths(cls_train_df)
cls_val_df = attach_roi_paths(cls_val_df)
cls_test_df = attach_roi_paths(cls_test_df)

for name, frame in [
    ("train", cls_train_df), ("validation", cls_val_df), ("test", cls_test_df)
]:
    if frame.roi_path.isna().any() or not frame.roi_path.map(Path).map(Path.exists).all():
        raise RuntimeError(f"{name} ROI split contains missing files.")
    frame.to_csv(MANIFEST_DIR / f"classification_{RUN_MODE}_{name}.csv", index=False)

print("ROI fallback rate:", classification_roi_manifest.roi_fallback_used.astype(bool).mean())
display(classification_roi_manifest[roi_columns].head())


## 18. ROI examples


In [ ]:
example_rows = classification_roi_manifest.sample(
    n=min(4, len(classification_roi_manifest)), random_state=SEED
)
fig, axes = plt.subplots(len(example_rows), 3, figsize=(12, 3.5 * len(example_rows)))
axes = np.atleast_2d(axes)
for row_number, (_, row) in enumerate(example_rows.iterrows()):
    original = load_rgb_pil(row.original_path)
    probability = predict_resized_segmentation(selected_segmenter, original)
    probability_original = np.asarray(
        Image.fromarray(probability.astype(np.float32), mode="F").resize(
            original.size, Image.Resampling.BILINEAR
        ), dtype=np.float32
    )
    mask = postprocess_binary_mask(probability_original)
    roi = load_rgb_pil(row.roi_path)
    axes[row_number, 0].imshow(original)
    axes[row_number, 0].set_title("Original")
    axes[row_number, 1].imshow(mask, cmap="gray")
    axes[row_number, 1].set_title("Predicted mask")
    axes[row_number, 2].imshow(roi)
    axes[row_number, 2].set_title(f"Padded ROI (fallback={bool(row.roi_fallback_used)})")
    for axis in axes[row_number]:
        axis.axis("off")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "roi_examples.png", dpi=180, bbox_inches="tight")
plt.show()


## 19. `tf.data` classification pipeline and moderate ROI augmentation


In [ ]:
classification_augmenter = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical", seed=SEED),
    layers.RandomRotation(0.06, fill_mode="reflect", seed=SEED + 1),
    layers.RandomZoom(height_factor=(-0.08, 0.08), width_factor=(-0.08, 0.08),
                      fill_mode="reflect", seed=SEED + 2),
    layers.RandomTranslation(0.05, 0.05, fill_mode="reflect", seed=SEED + 3),
    layers.RandomContrast(0.10, seed=SEED + 4),
    layers.RandomBrightness(0.08, value_range=(0.0, 255.0), seed=SEED + 5),
], name="classification_augmentation")


def decode_classification_image(path, target, training=False):
    image = tf.io.decode_image(
        tf.io.read_file(path), channels=3, expand_animations=False
    )
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, CLS_IMAGE_SIZE, method="bilinear")
    image = tf.cast(image, tf.float32)
    if training:
        image = classification_augmenter(image, training=True)
        image = tf.clip_by_value(image, 0.0, 255.0)
    image.set_shape((*CLS_IMAGE_SIZE, 3))
    return image, tf.cast(target, tf.float32)


def make_classification_dataset(frame, path_column="roi_path", training=False):
    ds = tf.data.Dataset.from_tensor_slices((
        frame[path_column].astype(str).values,
        frame["target"].astype(np.float32).values,
    ))
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(
        lambda path, target: decode_classification_image(path, target, training=training),
        num_parallel_calls=AUTOTUNE,
        deterministic=True,
    )
    ds = ds.batch(CLS_BATCH_SIZE, drop_remainder=False)
    return ds.prefetch(AUTOTUNE)


train_cls_ds = make_classification_dataset(cls_train_df, "roi_path", training=True)
val_cls_ds = make_classification_dataset(cls_val_df, "roi_path", training=False)
test_cls_ds = make_classification_dataset(cls_test_df, "roi_path", training=False)

example_images, example_targets = next(iter(val_cls_ds))
print("EfficientNet input dtype:", example_images.dtype)
print("EfficientNet input range:", float(tf.reduce_min(example_images)), "to", float(tf.reduce_max(example_images)))
assert float(tf.reduce_min(example_images)) >= 0.0
assert float(tf.reduce_max(example_images)) <= 255.0
print(
    "No extra preprocess_input call is used: tf.keras EfficientNetB0 includes its own Rescaling layer "
    "and expects float pixels in [0,255]."
)

train_counts = cls_train_df.target.value_counts().sort_index()
if set(train_counts.index) != {0, 1}:
    raise RuntimeError("Both classes must be present in the training split.")
weight_values = compute_class_weight(
    class_weight="balanced", classes=np.array([0, 1]), y=cls_train_df.target.values
)
CLASS_WEIGHTS = {0: float(weight_values[0]), 1: float(weight_values[1])}
print("Training class counts:", train_counts.to_dict())
print("Class weights calculated from TRAINING ONLY:", CLASS_WEIGHTS)


## 20. EfficientNetB0 transfer-learning classifier


In [ ]:
def build_efficientnet_classifier(
    input_shape=(*CLS_IMAGE_SIZE, 3),
    weights="imagenet",
    model_name="efficientnet_b0_melanoma",
):
    inputs = keras.Input(shape=input_shape, dtype=tf.float32, name="lesion_image")
    backbone = keras.applications.EfficientNetB0(
        include_top=False,
        weights=weights,
        input_shape=input_shape,
    )
    backbone.trainable = False
    features = backbone(inputs, training=False)
    x = layers.GlobalAveragePooling2D(name="cls_gap")(features)
    x = layers.Dropout(0.30, name="cls_dropout")(x)
    outputs = layers.Dense(
        1, activation="sigmoid", dtype="float32", name="malignancy_probability"
    )(x)
    return keras.Model(inputs, outputs, name=model_name)


def classification_metrics():
    return [
        keras.metrics.BinaryAccuracy(name="binary_accuracy"),
        keras.metrics.AUC(curve="ROC", name="roc_auc"),
        keras.metrics.AUC(curve="PR", name="pr_auc"),
    ]


def compile_classifier(model, learning_rate):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=keras.losses.BinaryCrossentropy(),
        metrics=classification_metrics(),
    )
    return model


def classifier_callbacks(checkpoint_path, patience=5):
    return [
        keras.callbacks.ModelCheckpoint(
            checkpoint_path,
            monitor="val_roc_auc",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_roc_auc",
            mode="max",
            patience=patience,
            restore_best_weights=False,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.3,
            patience=3,
            min_lr=1e-7,
            verbose=1,
        ),
    ]


classifier_preview = build_efficientnet_classifier(weights=None)
synthetic_cls_input = tf.zeros((1, *CLS_IMAGE_SIZE, 3), dtype=tf.float32)
synthetic_cls_output = classifier_preview(synthetic_cls_input, training=False)
assert synthetic_cls_output.shape == (1, 1)
assert synthetic_cls_output.dtype == tf.float32
print("EfficientNetB0 synthetic forward pass succeeded.")
print("Classifier parameters:", f"{classifier_preview.count_params():,}")
del classifier_preview, synthetic_cls_input, synthetic_cls_output
gc.collect()


## 21. Two-phase classifier training


In [ ]:
def unfreeze_efficientnet_top(model, last_n=CLS_FINE_TUNE_LAST_N):
    backbone = model.get_layer("efficientnetb0")
    backbone.trainable = True
    cutoff = max(0, len(backbone.layers) - int(last_n))
    for index, layer in enumerate(backbone.layers):
        layer.trainable = index >= cutoff and not isinstance(layer, layers.BatchNormalization)
    trainable_layers = sum(layer.trainable for layer in backbone.layers)
    print(f"Fine-tuning {trainable_layers} upper backbone layers; all BatchNormalization layers remain frozen.")
    return model


def predict_probabilities(model, frame, path_column):
    ds = make_classification_dataset(frame, path_column, training=False)
    probabilities = model.predict(ds, verbose=1).reshape(-1).astype(float)
    if len(probabilities) != len(frame):
        raise RuntimeError("Prediction count does not match manifest length.")
    return probabilities


def train_classifier_experiment(
    experiment_name,
    path_column,
    head_path,
    fine_path,
    final_path,
):
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)
    training_ds = make_classification_dataset(cls_train_df, path_column, training=True)
    validation_ds = make_classification_dataset(cls_val_df, path_column, training=False)

    model = compile_classifier(
        build_efficientnet_classifier(model_name=f"{experiment_name}_classifier"),
        CLS_HEAD_LEARNING_RATE,
    )
    start = time.perf_counter()
    head_history = model.fit(
        training_ds,
        validation_data=validation_ds,
        epochs=CLS_HEAD_EPOCHS,
        class_weight=CLASS_WEIGHTS,
        callbacks=classifier_callbacks(head_path),
        verbose=1,
    )
    head_elapsed = time.perf_counter() - start
    head_history_df = pd.DataFrame(head_history.history)
    head_history_df.index.name = "epoch"
    head_history_df.to_csv(RESULTS_DIR / f"{experiment_name}_head_history.csv")

    model = keras.models.load_model(head_path, compile=False)
    model = unfreeze_efficientnet_top(model)
    model = compile_classifier(model, CLS_FINE_TUNE_LEARNING_RATE)
    start = time.perf_counter()
    fine_history = model.fit(
        training_ds,
        validation_data=validation_ds,
        epochs=CLS_FINE_TUNE_EPOCHS,
        class_weight=CLASS_WEIGHTS,
        callbacks=classifier_callbacks(fine_path),
        verbose=1,
    )
    fine_elapsed = time.perf_counter() - start
    fine_history_df = pd.DataFrame(fine_history.history)
    fine_history_df.index.name = "epoch"
    fine_history_df.to_csv(RESULTS_DIR / f"{experiment_name}_fine_tune_history.csv")

    # Select head-only versus fine-tuned checkpoint using validation ROC-AUC only.
    checkpoint_scores = []
    for phase, checkpoint in [("head", head_path), ("fine_tuned", fine_path)]:
        candidate = keras.models.load_model(checkpoint, compile=False)
        val_prob = predict_probabilities(candidate, cls_val_df, path_column)
        checkpoint_scores.append((roc_auc_score(cls_val_df.target, val_prob), phase, checkpoint))
        del candidate
        gc.collect()
    checkpoint_scores.sort(reverse=True, key=lambda item: item[0])
    selected_auc, selected_phase, selected_checkpoint = checkpoint_scores[0]
    shutil.copy2(selected_checkpoint, final_path)
    print(f"{experiment_name}: selected {selected_phase} checkpoint from validation ROC-AUC={selected_auc:.4f}")

    return {
        "experiment": experiment_name,
        "path_column": path_column,
        "head_history": head_history_df,
        "fine_history": fine_history_df,
        "head_seconds": head_elapsed,
        "fine_tune_seconds": fine_elapsed,
        "selected_phase": selected_phase,
        "validation_roc_auc": selected_auc,
        "final_path": str(final_path),
    }


CLS_HEAD_PATH = MODELS_DIR / "efficientnet_b0_head_best.keras"
CLS_FINE_PATH = MODELS_DIR / "efficientnet_b0_finetuned_best.keras"
FINAL_CLASSIFIER_PATH = MODELS_DIR / "efficientnet_b0_final.keras"

roi_classifier_run = train_classifier_experiment(
    experiment_name="roi",
    path_column="roi_path",
    head_path=CLS_HEAD_PATH,
    fine_path=CLS_FINE_PATH,
    final_path=FINAL_CLASSIFIER_PATH,
)
final_classifier = keras.models.load_model(FINAL_CLASSIFIER_PATH, compile=False)


## 22. Classification learning curves


In [ ]:
def plot_classifier_histories(run, filename):
    head = run["head_history"].copy()
    fine = run["fine_history"].copy()
    head_epochs = np.arange(1, len(head) + 1)
    fine_epochs = np.arange(len(head) + 1, len(head) + len(fine) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for metric, axis, title in [
        ("loss", axes[0], "Classification loss"),
        ("roc_auc", axes[1], "Classification ROC-AUC"),
    ]:
        axis.plot(head_epochs, head[metric], label="head train")
        axis.plot(head_epochs, head[f"val_{metric}"], label="head validation")
        axis.plot(fine_epochs, fine[metric], label="fine-tune train")
        axis.plot(fine_epochs, fine[f"val_{metric}"], label="fine-tune validation")
        axis.axvline(len(head) + 0.5, color="black", linestyle="--", alpha=0.5)
        axis.set_title(title)
        axis.set_xlabel("Epoch")
        axis.grid(alpha=0.25)
        axis.legend()
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / filename, dpi=180, bbox_inches="tight")
    plt.show()


plot_classifier_histories(roi_classifier_run, "classification_learning_curves.png")


## 23. Validation-only threshold selection


In [ ]:
validation_probabilities = predict_probabilities(final_classifier, cls_val_df, "roi_path")
fpr, tpr, validation_thresholds = roc_curve(cls_val_df.target, validation_probabilities)
finite = np.isfinite(validation_thresholds)
youden_scores = tpr[finite] - fpr[finite]
youden_index = int(np.argmax(youden_scores))
CLASSIFICATION_THRESHOLD = float(validation_thresholds[finite][youden_index])

grid = np.linspace(0.0, 1.0, 1001)
threshold_rows = []
for threshold in grid:
    predicted = (validation_probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(cls_val_df.target, predicted, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if tp + fn else np.nan
    specificity = tn / (tn + fp) if tn + fp else np.nan
    threshold_rows.append((threshold, sensitivity, specificity))
threshold_table = pd.DataFrame(threshold_rows, columns=["threshold", "sensitivity", "specificity"])

high_sensitivity_candidates = threshold_table[threshold_table.sensitivity >= 0.90]
HIGH_SENSITIVITY_THRESHOLD = None
if len(high_sensitivity_candidates):
    best_high_sensitivity = high_sensitivity_candidates.sort_values(
        ["specificity", "threshold"], ascending=[False, False]
    ).iloc[0]
    HIGH_SENSITIVITY_THRESHOLD = float(best_high_sensitivity.threshold)

threshold_payload = {
    "selection_data": "validation_only",
    "criterion": "maximum_youden_j",
    "selected_threshold": CLASSIFICATION_THRESHOLD,
    "default_threshold": 0.5,
    "optional_high_sensitivity_threshold": HIGH_SENSITIVITY_THRESHOLD,
    "segmentation_threshold": SEG_THRESHOLD,
}
(MODELS_DIR / "classification_threshold.json").write_text(
    json.dumps(threshold_payload, indent=2), encoding="utf-8"
)
(RESULTS_DIR / "final_thresholds.json").write_text(
    json.dumps(threshold_payload, indent=2), encoding="utf-8"
)

print("Locked validation Youden-J threshold:", CLASSIFICATION_THRESHOLD)
print("Optional validation threshold targeting ≥90% sensitivity:", HIGH_SENSITIVITY_THRESHOLD)

plt.figure(figsize=(7, 5))
plt.plot(threshold_table.threshold, threshold_table.sensitivity, label="Sensitivity")
plt.plot(threshold_table.threshold, threshold_table.specificity, label="Specificity")
plt.axvline(CLASSIFICATION_THRESHOLD, linestyle="--", color="black", label="Selected threshold")
plt.xlabel("Classification threshold")
plt.ylabel("Metric")
plt.title("Validation threshold analysis")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "validation_threshold_analysis.png", dpi=180, bbox_inches="tight")
plt.show()


## 24. Held-out classification test evaluation


In [ ]:
def safe_roc_auc(y_true, probabilities):
    return float(roc_auc_score(y_true, probabilities)) if len(np.unique(y_true)) == 2 else np.nan


def classification_metrics_at_threshold(y_true, probabilities, threshold):
    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    predicted = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, predicted)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, predicted)),
        "sensitivity": float(tp / (tp + fn)) if tp + fn else np.nan,
        "specificity": float(tn / (tn + fp)) if tn + fp else np.nan,
        "precision": float(precision_score(y_true, predicted, zero_division=0)),
        "f1": float(f1_score(y_true, predicted, zero_division=0)),
        "roc_auc": safe_roc_auc(y_true, probabilities),
        "pr_auc": float(average_precision_score(y_true, probabilities)),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


test_probabilities = predict_probabilities(final_classifier, cls_test_df, "roi_path")
classification_results = pd.DataFrame([
    {"setting": "default_0.5", **classification_metrics_at_threshold(
        cls_test_df.target, test_probabilities, 0.5
    )},
    {"setting": "validation_youden_j", **classification_metrics_at_threshold(
        cls_test_df.target, test_probabilities, CLASSIFICATION_THRESHOLD
    )},
])
classification_results.to_csv(RESULTS_DIR / "classification_results.csv", index=False)
(RESULTS_DIR / "classification_results.json").write_text(
    classification_results.to_json(orient="records", indent=2), encoding="utf-8"
)
display(classification_results)

selected_test_predictions = (test_probabilities >= CLASSIFICATION_THRESHOLD).astype(int)
print(classification_report(
    cls_test_df.target,
    selected_test_predictions,
    labels=[0, 1],
    target_names=["Benign", "Malignant"],
    zero_division=0,
))


In [ ]:
test_fpr, test_tpr, _ = roc_curve(cls_test_df.target, test_probabilities)
precision_curve, recall_curve, _ = precision_recall_curve(cls_test_df.target, test_probabilities)
matrix = confusion_matrix(cls_test_df.target, selected_test_predictions, labels=[0, 1])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].plot(test_fpr, test_tpr, label=f"AUC={roc_auc_score(cls_test_df.target, test_probabilities):.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="grey")
axes[0].set(xlabel="False-positive rate", ylabel="True-positive rate", title="Test ROC curve")
axes[0].legend()

axes[1].plot(recall_curve, precision_curve,
             label=f"AP={average_precision_score(cls_test_df.target, test_probabilities):.3f}")
axes[1].set(xlabel="Recall", ylabel="Precision", title="Test precision-recall curve")
axes[1].legend()

image = axes[2].imshow(matrix, cmap="Blues")
for i in range(2):
    for j in range(2):
        axes[2].text(j, i, matrix[i, j], ha="center", va="center", fontsize=13)
axes[2].set_xticks([0, 1], ["Benign", "Malignant"])
axes[2].set_yticks([0, 1], ["Benign", "Malignant"])
axes[2].set(xlabel="Predicted", ylabel="True", title="Test confusion matrix")
fig.colorbar(image, ax=axes[2], fraction=0.046)
for axis in axes[:2]:
    axis.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "classification_test_evaluation.png", dpi=180, bbox_inches="tight")
plt.show()


## 25. Optional full-image versus ROI ablation


In [ ]:
ablation_rows = []
roi_test_selected = classification_metrics_at_threshold(
    cls_test_df.target, test_probabilities, CLASSIFICATION_THRESHOLD
)
ablation_rows.append({"input": "U-Net-derived lesion ROI", **roi_test_selected})

if RUN_ABLATION:
    FULL_HEAD_PATH = MODELS_DIR / "efficientnet_b0_full_image_head_best.keras"
    FULL_FINE_PATH = MODELS_DIR / "efficientnet_b0_full_image_finetuned_best.keras"
    FULL_FINAL_PATH = MODELS_DIR / "efficientnet_b0_full_image_final.keras"
    full_classifier_run = train_classifier_experiment(
        experiment_name="full_image",
        path_column="original_path",
        head_path=FULL_HEAD_PATH,
        fine_path=FULL_FINE_PATH,
        final_path=FULL_FINAL_PATH,
    )
    full_classifier = keras.models.load_model(FULL_FINAL_PATH, compile=False)
    full_val_prob = predict_probabilities(full_classifier, cls_val_df, "original_path")
    full_fpr, full_tpr, full_thresholds = roc_curve(cls_val_df.target, full_val_prob)
    valid = np.isfinite(full_thresholds)
    full_threshold = float(full_thresholds[valid][np.argmax(full_tpr[valid] - full_fpr[valid])])
    full_test_prob = predict_probabilities(full_classifier, cls_test_df, "original_path")
    ablation_rows.append({
        "input": "Full dermoscopic image",
        **classification_metrics_at_threshold(cls_test_df.target, full_test_prob, full_threshold),
    })
    plot_classifier_histories(full_classifier_run, "full_image_learning_curves.png")
else:
    print("Ablation is disabled. Set RUN_ABLATION=True before training to run the fair full-image comparison.")

ablation_comparison = pd.DataFrame(ablation_rows)
ablation_comparison.to_csv(RESULTS_DIR / "roi_vs_full_image_ablation.csv", index=False)
display(ablation_comparison[[
    "input", "roc_auc", "pr_auc", "sensitivity", "specificity", "f1", "threshold"
]])


## 26. Grad-CAM and the reusable end-to-end inference function


In [ ]:
def get_efficientnet_backbone(classifier):
    for layer in classifier.layers:
        if isinstance(layer, keras.Model) and "efficientnet" in layer.name.lower():
            return layer
    raise ValueError("EfficientNet backbone was not found inside the classifier.")


def find_last_spatial_layer(backbone):
    for layer in reversed(backbone.layers):
        try:
            output_shape = tuple(layer.output.shape)
        except Exception:
            continue
        if len(output_shape) == 4:
            return layer
    raise ValueError("No four-dimensional convolutional feature layer was found.")


def make_gradcam_heatmap(classifier, image_batch, target_class=1):
    backbone = get_efficientnet_backbone(classifier)
    last_spatial_layer = find_last_spatial_layer(backbone)
    feature_model = keras.Model(backbone.input, last_spatial_layer.output)

    image_batch = tf.cast(image_batch, tf.float32)
    with tf.GradientTape() as tape:
        feature_maps = feature_model(image_batch, training=False)
        tape.watch(feature_maps)
        pooled = classifier.get_layer("cls_gap")(feature_maps)
        dropped = classifier.get_layer("cls_dropout")(pooled, training=False)
        probability = classifier.get_layer("malignancy_probability")(dropped)
        target_score = probability[:, 0] if int(target_class) == 1 else (1.0 - probability[:, 0])

    gradients = tape.gradient(target_score, feature_maps)
    if gradients is None:
        raise RuntimeError("Grad-CAM gradients could not be calculated.")
    weights = tf.reduce_mean(gradients, axis=(1, 2), keepdims=True)
    heatmap = tf.reduce_sum(weights * feature_maps, axis=-1)
    heatmap = tf.nn.relu(heatmap)
    maximum = tf.reduce_max(heatmap, axis=(1, 2), keepdims=True)
    heatmap = heatmap / tf.maximum(maximum, tf.keras.backend.epsilon())
    return heatmap[0].numpy().astype(np.float32), last_spatial_layer.name


def heatmap_overlay(image, heatmap, alpha=0.40):
    image_array = np.asarray(image.convert("RGB") if isinstance(image, Image.Image) else image).astype(np.uint8)
    resized_heatmap = np.asarray(
        Image.fromarray(np.asarray(heatmap, dtype=np.float32), mode="F").resize(
            (image_array.shape[1], image_array.shape[0]), Image.Resampling.BILINEAR
        ), dtype=np.float32
    )
    coloured = plt.get_cmap("jet")(np.clip(resized_heatmap, 0.0, 1.0))[..., :3]
    coloured = (coloured * 255).astype(np.uint8)
    overlay = np.clip((1.0 - alpha) * image_array + alpha * coloured, 0, 255).astype(np.uint8)
    return overlay


def segmentation_overlay(image, binary_mask, alpha=0.35):
    image_array = np.asarray(image.convert("RGB") if isinstance(image, Image.Image) else image).astype(np.uint8)
    mask = np.asarray(binary_mask).squeeze().astype(bool)
    if mask.shape != image_array.shape[:2]:
        mask = np.asarray(
            Image.fromarray(mask.astype(np.uint8) * 255).resize(
                (image_array.shape[1], image_array.shape[0]), Image.Resampling.NEAREST
            )
        ) > 127
    colour = np.zeros_like(image_array)
    colour[..., 0] = 255
    return np.where(
        mask[..., None],
        np.clip((1.0 - alpha) * image_array + alpha * colour, 0, 255),
        image_array,
    ).astype(np.uint8)


# Synthetic Grad-CAM test with a small network using the same named classifier head.
def build_gradcam_smoke_model():
    backbone_input = keras.Input((32, 32, 3))
    x = layers.Conv2D(4, 3, activation="relu", name="smoke_conv")(backbone_input)
    backbone = keras.Model(backbone_input, x, name="efficientnet_smoke")
    inputs = keras.Input((32, 32, 3), name="lesion_image")
    features = backbone(inputs)
    pooled = layers.GlobalAveragePooling2D(name="cls_gap")(features)
    dropped = layers.Dropout(0.0, name="cls_dropout")(pooled)
    outputs = layers.Dense(1, activation="sigmoid", dtype="float32",
                           name="malignancy_probability")(dropped)
    return keras.Model(inputs, outputs)


smoke_model = build_gradcam_smoke_model()
smoke_heatmap, smoke_layer = make_gradcam_heatmap(
    smoke_model, tf.zeros((1, 32, 32, 3), dtype=tf.float32), target_class=1
)
assert smoke_heatmap.ndim == 2 and np.isfinite(smoke_heatmap).all()
print("Grad-CAM synthetic check passed using layer:", smoke_layer)
del smoke_model, smoke_heatmap
gc.collect()


In [ ]:
def validate_input_image(image):
    if image is None:
        raise ValueError("No image was supplied.")
    if isinstance(image, (str, Path)):
        image = load_rgb_pil(image)
    elif isinstance(image, np.ndarray):
        if image.size == 0:
            raise ValueError("The supplied image array is empty.")
        image = Image.fromarray(np.asarray(image).astype(np.uint8)).convert("RGB")
    elif isinstance(image, Image.Image):
        image = image.convert("RGB")
    else:
        raise TypeError("Input must be a PIL image, NumPy array, or image path.")
    if min(image.size) < 32:
        raise ValueError("Image is too small. Both dimensions must be at least 32 pixels.")
    return image


def predict_melanoma(image):
    """Run the shared end-to-end pipeline used by analyses and the Gradio app."""
    original = validate_input_image(image)

    seg_resized = original.resize(
        (SEG_IMAGE_SIZE[1], SEG_IMAGE_SIZE[0]), Image.Resampling.BILINEAR
    )
    seg_input = np.asarray(seg_resized, dtype=np.float32)[None, ...] / 255.0
    seg_probability_small = selected_segmenter.predict(seg_input, verbose=0)[0, ..., 0]
    seg_probability = np.asarray(
        Image.fromarray(seg_probability_small.astype(np.float32), mode="F").resize(
            original.size, Image.Resampling.BILINEAR
        ), dtype=np.float32
    )
    binary_mask = postprocess_binary_mask(seg_probability, SEG_THRESHOLD)
    roi_array, bbox, fallback_used, fallback_reason = extract_lesion_roi(original, binary_mask)
    roi_image = Image.fromarray(roi_array)

    classifier_resized = roi_image.resize(
        (CLS_IMAGE_SIZE[1], CLS_IMAGE_SIZE[0]), Image.Resampling.BILINEAR
    )
    classifier_input = np.asarray(classifier_resized, dtype=np.float32)[None, ...]
    probability = float(final_classifier.predict(classifier_input, verbose=0)[0, 0])
    predicted_class = int(probability >= CLASSIFICATION_THRESHOLD)
    heatmap, gradcam_layer = make_gradcam_heatmap(
        final_classifier, classifier_input, target_class=predicted_class
    )
    gradcam_image = heatmap_overlay(roi_image, heatmap)

    return {
        "original_image": np.asarray(original),
        "predicted_segmentation_mask": binary_mask.astype(np.uint8),
        "segmentation_overlay": segmentation_overlay(original, binary_mask),
        "roi_image": roi_array,
        "bounding_box": tuple(int(value) for value in bbox),
        "malignancy_probability": probability,
        "classification_threshold": float(CLASSIFICATION_THRESHOLD),
        "predicted_class": predicted_class,
        "predicted_label": "Malignant" if predicted_class else "Benign",
        "gradcam_heatmap": heatmap,
        "gradcam_overlay": gradcam_image,
        "gradcam_target": "predicted malignant class" if predicted_class else "predicted benign class",
        "gradcam_layer": gradcam_layer,
        "roi_fallback_used": bool(fallback_used),
        "roi_fallback_reason": fallback_reason,
    }


print("End-to-end inference function defined. It is reused below and by Gradio.")


### Correct interpretation of Grad-CAM

Grad-CAM highlights ROI regions that most influenced the model's selected class score. It can help identify attention to lesion structure, hair, rulers, reflections, or background. It **does not prove clinical causality**, explain the biological reason a lesion is melanoma, or validate that the highlighted region is medically appropriate.


## 27. Representative TP, TN, FP, and FN cases


In [ ]:
test_case_table = cls_test_df[[
    "image_name", "original_path", "target", "roi_path", "roi_fallback_used"
]].copy()
test_case_table["probability"] = test_probabilities
test_case_table["prediction"] = selected_test_predictions
test_case_table["error_type"] = np.select(
    [
        (test_case_table.target == 1) & (test_case_table.prediction == 1),
        (test_case_table.target == 0) & (test_case_table.prediction == 0),
        (test_case_table.target == 0) & (test_case_table.prediction == 1),
        (test_case_table.target == 1) & (test_case_table.prediction == 0),
    ],
    ["true_positive", "true_negative", "false_positive", "false_negative"],
    default="unknown",
)
test_case_table.to_csv(RESULTS_DIR / "classification_test_predictions.csv", index=False)


def display_end_to_end_case(row, title):
    result = predict_melanoma(row.original_path)
    fig, axes = plt.subplots(1, 5, figsize=(17, 3.5))
    axes[0].imshow(result["original_image"])
    axes[0].set_title("Original")
    axes[1].imshow(result["predicted_segmentation_mask"], cmap="gray")
    axes[1].set_title("Segmentation")
    axes[2].imshow(result["roi_image"])
    axes[2].set_title("Lesion ROI")
    axes[3].imshow(result["gradcam_heatmap"], cmap="jet")
    axes[3].set_title("Grad-CAM")
    axes[4].imshow(result["gradcam_overlay"])
    axes[4].set_title("Grad-CAM overlay")
    for axis in axes:
        axis.axis("off")
    fig.suptitle(
        f"{title} | true={int(row.target)} | predicted={result['predicted_class']} | "
        f"model probability of malignancy={result['malignancy_probability']:.3f}",
        y=1.03,
    )
    plt.tight_layout()
    plt.show()


for case_type in ["true_positive", "true_negative", "false_positive", "false_negative"]:
    available = test_case_table[test_case_table.error_type == case_type]
    if len(available):
        display_end_to_end_case(available.iloc[0], case_type.replace("_", " ").title())
    else:
        print(f"No {case_type.replace('_', ' ')} existed in this test run.")


## 28. Error analysis


In [ ]:
error_cases = test_case_table[
    test_case_table.error_type.isin(["false_positive", "false_negative"])
].copy()

if len(error_cases):
    for _, row in error_cases.head(6).iterrows():
        display_end_to_end_case(row, row.error_type.replace("_", " ").title())
else:
    print("No false-positive or false-negative cases were available to display in this run.")


Possible causes to inspect in errors include poor lesion segmentation, hair crossing the lesion, ruler markings, specular reflections, low contrast, lighting or colour variation, and unusual lesion morphology. Grad-CAM can reveal model attention but does not establish the clinical cause of an error. Each error should be reviewed against the source image, mask quality, ROI fallback flag, probability, and acquisition conditions.


## 29. Skin-tone bias, fairness limitations, and an external evaluation interface


The ISIC datasets used here do not provide sufficiently complete and reliable Fitzpatrick or Monk skin-tone labels for a valid direct subgroup analysis. Therefore this notebook:

- does not invent skin-tone groups;
- does not treat average image brightness as skin tone;
- does not claim that artificially darkening images resolves demographic bias.

Darker skin tones are underrepresented in many dermatology AI datasets, which limits generalisability and can reinforce health inequities. A future validation dataset should include expert-annotated skin-tone metadata, representative dermoscopic acquisition, and adequate subgroup sample sizes.

Datasets such as Diverse Dermatology Images include real skin-tone annotations, but their clinical photography differs from the dermoscopic ISIC modality. They should not simply be mixed into the main training set without accounting for domain shift.

Future mitigation should include collecting more dermoscopic data across skin tones, expert-annotated subgroup metadata, external subgroup validation, and—only where genuine labels exist—reweighting or oversampling underrepresented groups.


In [ ]:
def fairness_metrics_by_group(
    data,
    group_column="skin_tone_group",
    target_column="ground_truth_label",
    probability_column="model_probability",
    threshold=CLASSIFICATION_THRESHOLD,
    minimum_auc_samples=10,
):
    required = {group_column, target_column, probability_column}
    missing = required - set(data.columns)
    if missing:
        raise ValueError(f"Fairness table is missing columns: {sorted(missing)}")

    rows = []
    for group, subset in data.groupby(group_column, dropna=False):
        y_true = subset[target_column].astype(int).values
        probability = subset[probability_column].astype(float).values
        predicted = (probability >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
        meaningful_auc = len(subset) >= minimum_auc_samples and len(np.unique(y_true)) == 2
        rows.append({
            "skin_tone_group": group,
            "sample_count": len(subset),
            "sensitivity": tp / (tp + fn) if tp + fn else np.nan,
            "specificity": tn / (tn + fp) if tn + fp else np.nan,
            "accuracy": accuracy_score(y_true, predicted),
            "f1": f1_score(y_true, predicted, zero_division=0),
            "roc_auc": roc_auc_score(y_true, probability) if meaningful_auc else np.nan,
            "auc_interpretable": bool(meaningful_auc),
        })
    return pd.DataFrame(rows)


def evaluate_external_fairness_dataset(csv_path, output_path=None):
    external = pd.read_csv(csv_path)
    required = {"image_path", "skin_tone_group", "ground_truth_label"}
    missing = required - set(external.columns)
    if missing:
        raise ValueError(
            "External fairness CSV must contain image_path, skin_tone_group, and ground_truth_label. "
            f"Missing: {sorted(missing)}"
        )
    probabilities = []
    failures = []
    for index, row in tqdm(external.iterrows(), total=len(external), desc="External fairness inference"):
        try:
            probabilities.append(predict_melanoma(row.image_path)["malignancy_probability"])
        except Exception as exc:
            probabilities.append(np.nan)
            failures.append({"row": index, "image_path": row.image_path, "error": repr(exc)})
    external["model_probability"] = probabilities
    valid = external.dropna(subset=["model_probability"]).copy()
    results = fairness_metrics_by_group(valid)
    if output_path is not None:
        output_path = Path(output_path)
        results.to_csv(output_path, index=False)
        pd.DataFrame(failures).to_csv(output_path.with_name(output_path.stem + "_failures.csv"), index=False)
    return results, pd.DataFrame(failures)


print(
    "Optional interface ready. Supply a genuine external CSV with: "
    "image_path, skin_tone_group, ground_truth_label."
)


## 30. Complete Gradio research demonstration


In [ ]:
import gradio as gr


def gradio_predict(image):
    result = predict_melanoma(image)
    probability_text = f"{100.0 * result['malignancy_probability']:.2f}%"
    prediction_text = (
        f"{result['predicted_label']} at the locked validation threshold "
        f"({result['classification_threshold']:.3f})"
    )
    details = {
        "bounding_box": result["bounding_box"],
        "roi_fallback_used": result["roi_fallback_used"],
        "roi_fallback_reason": result["roi_fallback_reason"],
        "gradcam_target": result["gradcam_target"],
        "disclaimer": (
            "Research/educational prototype only. This tool is not a medical diagnosis and "
            "should not replace assessment by a qualified healthcare professional."
        ),
    }
    return (
        result["original_image"],
        result["segmentation_overlay"],
        result["roi_image"],
        result["gradcam_overlay"],
        probability_text,
        prediction_text,
        details,
    )


demo = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Image(type="pil", label="Upload a dermoscopic image"),
    outputs=[
        gr.Image(label="1. Original image"),
        gr.Image(label="2. Segmentation overlay"),
        gr.Image(label="3. Lesion ROI"),
        gr.Image(label="4. Grad-CAM overlay"),
        gr.Textbox(label="5. Model probability of malignancy"),
        gr.Textbox(label="6. Thresholded model prediction"),
        gr.JSON(label="Pipeline details"),
    ],
    title="AI-Assisted Dermoscopic Melanoma Analysis",
    description=(
        "Research/educational prototype only. This tool is not a medical diagnosis and should "
        "not replace assessment by a qualified healthcare professional."
    ),
    flagging_mode="never",
)

if LAUNCH_GRADIO:
    demo.launch(share=True, debug=False)
else:
    print("Gradio app constructed. Set LAUNCH_GRADIO=True to launch it with share=True.")


## 31. Export a Hugging Face Spaces deployment package


In [ ]:
APP_SOURCE = r"""
import json
from pathlib import Path

import gradio as gr
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image
from scipy import ndimage

ROOT = Path(__file__).resolve().parent
segmenter = tf.keras.models.load_model(ROOT / "selected_segmenter.keras", compile=False)
classifier = tf.keras.models.load_model(ROOT / "efficientnet_b0_final.keras", compile=False)
thresholds = json.loads((ROOT / "classification_threshold.json").read_text())
SEG_THRESHOLD = float(thresholds.get("segmentation_threshold", 0.5))
CLS_THRESHOLD = float(thresholds["selected_threshold"])
SEG_SIZE = (256, 256)
CLS_SIZE = (224, 224)


def postprocess_mask(probability):
    mask = np.asarray(probability) >= SEG_THRESHOLD
    if not np.any(mask):
        return mask.astype(np.uint8)
    labels, count = ndimage.label(mask)
    if count > 1:
        sizes = ndimage.sum(mask, labels, index=np.arange(1, count + 1))
        mask = labels == (int(np.argmax(sizes)) + 1)
    return ndimage.binary_fill_holes(mask).astype(np.uint8)


def extract_roi(image, mask):
    array = np.asarray(image.convert("RGB"))
    fraction = float(mask.mean())
    height, width = mask.shape
    if not np.any(mask) or fraction < 0.001 or fraction > 0.95:
        return array, (0, 0, width, height), True
    ys, xs = np.where(mask)
    x0, x1 = int(xs.min()), int(xs.max()) + 1
    y0, y1 = int(ys.min()), int(ys.max()) + 1
    px = max(1, int((x1 - x0) * 0.10))
    py = max(1, int((y1 - y0) * 0.10))
    x0, x1 = max(0, x0 - px), min(width, x1 + px)
    y0, y1 = max(0, y0 - py), min(height, y1 + py)
    return array[y0:y1, x0:x1], (x0, y0, x1, y1), False


def backbone_and_last_layer(model):
    backbone = next(
        layer for layer in model.layers
        if isinstance(layer, tf.keras.Model) and "efficientnet" in layer.name.lower()
    )
    spatial = next(layer for layer in reversed(backbone.layers) if len(layer.output.shape) == 4)
    return backbone, spatial


def gradcam(image_batch, target_class):
    backbone, spatial = backbone_and_last_layer(classifier)
    feature_model = tf.keras.Model(backbone.input, spatial.output)
    with tf.GradientTape() as tape:
        features = feature_model(image_batch, training=False)
        tape.watch(features)
        x = classifier.get_layer("cls_gap")(features)
        x = classifier.get_layer("cls_dropout")(x, training=False)
        probability = classifier.get_layer("malignancy_probability")(x)[:, 0]
        score = probability if target_class else 1.0 - probability
    gradients = tape.gradient(score, features)
    weights = tf.reduce_mean(gradients, axis=(1, 2), keepdims=True)
    heatmap = tf.nn.relu(tf.reduce_sum(weights * features, axis=-1))[0]
    heatmap = heatmap / tf.maximum(tf.reduce_max(heatmap), tf.keras.backend.epsilon())
    return heatmap.numpy()


def overlay(image, heatmap, alpha=0.4):
    array = np.asarray(image.convert("RGB"), dtype=np.uint8)
    heatmap = np.asarray(
        Image.fromarray(heatmap.astype(np.float32), mode="F").resize(
            (array.shape[1], array.shape[0]), Image.Resampling.BILINEAR
        )
    )
    colour = (plt.get_cmap("jet")(np.clip(heatmap, 0, 1))[..., :3] * 255).astype(np.uint8)
    return np.clip((1 - alpha) * array + alpha * colour, 0, 255).astype(np.uint8)


def predict(image):
    if image is None:
        raise gr.Error("Please upload a dermoscopic image.")
    original = image.convert("RGB")
    if min(original.size) < 32:
        raise gr.Error("Both image dimensions must be at least 32 pixels.")
    seg_input = np.asarray(
        original.resize((SEG_SIZE[1], SEG_SIZE[0]), Image.Resampling.BILINEAR),
        dtype=np.float32,
    )[None] / 255.0
    probability_small = segmenter.predict(seg_input, verbose=0)[0, ..., 0]
    probability = np.asarray(
        Image.fromarray(probability_small.astype(np.float32), mode="F").resize(
            original.size, Image.Resampling.BILINEAR
        )
    )
    mask = postprocess_mask(probability)
    roi_array, bbox, fallback = extract_roi(original, mask)
    roi = Image.fromarray(roi_array)
    cls_input = np.asarray(
        roi.resize((CLS_SIZE[1], CLS_SIZE[0]), Image.Resampling.BILINEAR),
        dtype=np.float32,
    )[None]
    malignant_probability = float(classifier.predict(cls_input, verbose=0)[0, 0])
    predicted = int(malignant_probability >= CLS_THRESHOLD)
    heatmap = gradcam(tf.convert_to_tensor(cls_input), predicted)
    gradcam_overlay = overlay(roi, heatmap)

    original_array = np.asarray(original)
    red = np.zeros_like(original_array)
    red[..., 0] = 255
    seg_overlay = np.where(
        mask[..., None].astype(bool),
        np.clip(0.65 * original_array + 0.35 * red, 0, 255),
        original_array,
    ).astype(np.uint8)
    label = "Malignant" if predicted else "Benign"
    details = {
        "bounding_box": bbox,
        "roi_fallback_used": fallback,
        "threshold": CLS_THRESHOLD,
        "disclaimer": "Research/educational prototype only; not a medical diagnosis.",
    }
    return (
        np.asarray(original), seg_overlay, roi_array, gradcam_overlay,
        f"{100 * malignant_probability:.2f}%", f"{label} at threshold {CLS_THRESHOLD:.3f}", details,
    )


demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil", label="Dermoscopic image"),
    outputs=[
        gr.Image(label="Original"), gr.Image(label="Segmentation overlay"),
        gr.Image(label="Lesion ROI"), gr.Image(label="Grad-CAM overlay"),
        gr.Textbox(label="Model probability of malignancy"),
        gr.Textbox(label="Thresholded model prediction"), gr.JSON(label="Details"),
    ],
    title="AI-Assisted Dermoscopic Melanoma Analysis",
    description=(
        "Research/educational prototype only. This tool is not a medical diagnosis and should "
        "not replace assessment by a qualified healthcare professional."
    ),
    flagging_mode="never",
)

if __name__ == "__main__":
    demo.launch()
"""

REQUIREMENTS_SOURCE = "\n".join([
    "tensorflow>=2.16,<2.21",
    "gradio>=5,<7",
    "numpy<2.3",
    "pillow>=10",
    "scipy>=1.10",
    "matplotlib>=3.8",
    "socksio>=1.0",
    "",
])

README_SOURCE = """# AI-Assisted Dermoscopic Melanoma Analysis

Research/educational prototype only. This Space is not a medical diagnosis and should not replace assessment by a qualified healthcare professional.

The app loads the selected U-Net, final EfficientNetB0 classifier, and locked validation threshold, then runs segmentation, padded ROI extraction, malignancy-probability estimation, and Grad-CAM.
"""

ast.parse(APP_SOURCE)
DEPLOYMENT_DIR.mkdir(parents=True, exist_ok=True)
(DEPLOYMENT_DIR / "app.py").write_text(APP_SOURCE, encoding="utf-8")
(DEPLOYMENT_DIR / "requirements.txt").write_text(REQUIREMENTS_SOURCE, encoding="utf-8")
(DEPLOYMENT_DIR / "README.md").write_text(README_SOURCE, encoding="utf-8")
shutil.copy2(SELECTED_SEGMENTER_PATH, DEPLOYMENT_DIR / "selected_segmenter.keras")
shutil.copy2(FINAL_CLASSIFIER_PATH, DEPLOYMENT_DIR / "efficientnet_b0_final.keras")
shutil.copy2(MODELS_DIR / "classification_threshold.json", DEPLOYMENT_DIR / "classification_threshold.json")
shutil.copy2(RESULTS_DIR / "pipeline_config.json", DEPLOYMENT_DIR / "pipeline_config.json")

deployment_zip = shutil.make_archive(
    str(RESULTS_DIR / "Melanoma_AI_HF_Space"), "zip", DEPLOYMENT_DIR
)
print("Deployment files exported:", DEPLOYMENT_DIR)
print("Deployment ZIP:", deployment_zip)


## 32. Fresh-runtime model reload test


In [ ]:
def reload_saved_pipeline():
    segmenter = keras.models.load_model(
        MODELS_DIR / "selected_segmenter.keras",
        custom_objects=CUSTOM_OBJECTS,
        compile=False,
    )
    classifier = keras.models.load_model(
        MODELS_DIR / "efficientnet_b0_final.keras",
        compile=False,
    )
    threshold_data = json.loads(
        (MODELS_DIR / "classification_threshold.json").read_text(encoding="utf-8")
    )
    return segmenter, classifier, float(threshold_data["selected_threshold"])


reloaded_segmenter, reloaded_classifier, reloaded_threshold = reload_saved_pipeline()
assert reloaded_segmenter(tf.zeros((1, *SEG_IMAGE_SIZE, 3)), training=False).shape == (1, *SEG_IMAGE_SIZE, 1)
assert reloaded_classifier(tf.zeros((1, *CLS_IMAGE_SIZE, 3)), training=False).shape == (1, 1)
assert 0.0 <= reloaded_threshold <= 1.0
print("Saved models reloaded and both synthetic forward passes succeeded.")
del reloaded_segmenter, reloaded_classifier
gc.collect()


## 33. Final execution checklist

- [ ] GPU detected and mixed precision policy printed.
- [ ] Official 2018 image/mask IDs matched exactly (2,594 cases).
- [ ] Segmentation train/validation/test IDs have zero overlap.
- [ ] Standard U-Net and Attention U-Net trained under equivalent conditions.
- [ ] Segmenter selected from validation Dice/IoU before test evaluation.
- [ ] Official 2020 duplicate-copy IDs removed before splitting.
- [ ] Classification split is patient-level with zero patient overlap.
- [ ] ROI crops generated once in batched inference.
- [ ] Class weights calculated from training only.
- [ ] Classifier threshold selected from validation only.
- [ ] Test results reported at both 0.5 and the locked validation threshold.
- [ ] False positives/negatives and Grad-CAM reviewed.
- [ ] Fairness limitations stated without inventing skin-tone labels.
- [ ] Models reloaded successfully from `.keras` files.
- [ ] Gradio app launched with `share=True`.

### Main saved files

**Models**

- `models/standard_unet_best.keras`
- `models/attention_unet_best.keras`
- `models/selected_segmenter.keras`
- `models/efficientnet_b0_head_best.keras`
- `models/efficientnet_b0_finetuned_best.keras`
- `models/efficientnet_b0_final.keras`
- `models/classification_threshold.json`

**Results and reproducibility**

- configuration JSON, manifests, split CSVs, histories, thresholds, comparison tables, test predictions, evaluation plots, and a Hugging Face Spaces ZIP under `results/`.


## References

1. ISIC Challenge. [Official challenge datasets and download links](https://challenge.isic-archive.com/data/).
2. Codella, N. C. F., et al. (2019). *Skin Lesion Analysis Toward Melanoma Detection 2018: A Challenge Hosted by the International Skin Imaging Collaboration*. arXiv:1902.03368.
3. Rotemberg, V., et al. (2021). *A patient-centric dataset of images and metadata for identifying melanomas using clinical context*. Scientific Data, 8, 34. https://doi.org/10.1038/s41597-021-00815-z
4. Ronneberger, O., Fischer, P., & Brox, T. (2015). *U-Net: Convolutional Networks for Biomedical Image Segmentation*. MICCAI. https://doi.org/10.1007/978-3-319-24574-4_28
5. Oktay, O., et al. (2018). *Attention U-Net: Learning Where to Look for the Pancreas*. arXiv:1804.03999.
6. Tan, M., & Le, Q. (2019). *EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks*. ICML.
7. Selvaraju, R. R., et al. (2017). *Grad-CAM: Visual Explanations from Deep Networks via Gradient-Based Localization*. ICCV. https://doi.org/10.1109/ICCV.2017.74
8. TensorFlow. [EfficientNetB0 API: preprocessing is included in the model](https://www.tensorflow.org/api_docs/python/tf/keras/applications/EfficientNetB0).
